# Stored but Not Scored: reproduction

CPU only, about ten minutes. Attach the competition dataset (`gemma-4-developer-agent`). The notebook writes the measurement scripts, the frozen legacy system and the production retriever to `/kaggle/working/repro`, runs them on the 129 public tasks, and checks the numbers reported in the paper.

`hints_text` and `test_patch` are never read. `patch` is used for scoring only. The agent runs (Table 5) need Docker and a local model server and are not reproduced here.

In [ ]:
import glob, os, pathlib
hits = glob.glob('/kaggle/input/**/tasks.jsonl', recursive=True)
assert hits, 'attach the competition dataset'
DATA = str(pathlib.Path(hits[0]).parent)
W = '/kaggle/working/repro'
for d in ('semdag', 'legacy_semdag', 'phase1', 'phase4', 'out', 'out_keep_tests', 'out_legacy_gate', 'scratch'):
    os.makedirs(f'{W}/{d}', exist_ok=True)
os.environ.update(P4_DATA=DATA, P4_SEMDAG=f'{W}/semdag', P4_SEMDAG_LEGACY=f'{W}/legacy_semdag', P4_PHASE1=f'{W}/phase1',
                  P4_OUT=f'{W}/out', P4_SCRATCH=f'{W}/scratch', P4_COMPLETIONS=f'{W}/none', P4_PROCS='4', PYTHONDONTWRITEBYTECODE='1')
print(DATA)

## Legacy SemDAG (frozen copy of the system as first shipped)

In [ ]:
%%writefile /kaggle/working/repro/legacy_semdag/cache.py
"""SharedContainerCache: one namespace per container ("one probe per container").
All pipeline stages read/write here; the orchestrating agent reads only the
compact results, never the repo text — context is preserved by construction.

Layout:  <root>/<container_id>/{snapshot.json, semdag_ir.json, vulcan_ranking.json, meta.json}
Default root: $SEMDAG_CACHE or ~/.cache/semdag
"""
import os, json, time, hashlib

def cache_root() -> str:
    return os.environ.get("SEMDAG_CACHE", os.path.expanduser("~/.cache/semdag"))

def container_dir(container_id: str) -> str:
    d = os.path.join(cache_root(), container_id)
    os.makedirs(d, exist_ok=True)
    return d

def init_container(container_id: str, target: str) -> dict:
    d = container_dir(container_id)
    meta = {"container_id": container_id, "target": os.path.abspath(target),
            "created": time.time(), "stages": {}}
    write(container_id, "meta.json", meta)
    return meta

def path(container_id: str, name: str) -> str:
    return os.path.join(container_dir(container_id), name)

def write(container_id: str, name: str, obj) -> str:
    p = path(container_id, name)
    tmp = p + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, sort_keys=True, separators=(",", ":"))
    os.replace(tmp, p)  # atomic
    return p

def read(container_id: str, name: str):
    with open(path(container_id, name)) as f:
        return json.load(f)

def mark_stage(container_id: str, stage: str, info: dict):
    meta = read(container_id, "meta.json")
    meta.setdefault("stages", {})[stage] = {"at": time.time(), **info}
    write(container_id, "meta.json", meta)

def sha256_file(fp: str) -> str:
    h = hashlib.sha256()
    with open(fp, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 16), b""):
            h.update(chunk)
    return h.hexdigest()


In [ ]:
%%writefile /kaggle/working/repro/legacy_semdag/ppr_scorer.py
"""PurePythonPPRScorer — zero-dependency vulnerability-hub scorer.

Stdlib only (collections). No pip, no native compilation, no tree-sitter at
call time: it consumes the cached SemDAG-IR (+ optional taint seed) and runs
Personalized PageRank on the reversed call graph. Because tree-sitter already
produced and cached the IR, this scorer runs instantly inside any offline
sandbox container.

Typical use: a test fails or an issue is filed -> extract the traceback frame's
FUNCTION NAME -> `scorer.score_by_names([name])` -> top vulnerability hubs,
instantly. The seed can be function names, explicit node ids, or a raw weight
vector (e.g. the taint seed), and bias vectors can be layered in.
"""
from collections import defaultdict

D = 0.85
ITERS = 80


class PurePythonPPRScorer:
    def __init__(self, ir: dict, taint: dict | None = None):
        self.meta = {n["id"]: n for n in ir["nodes"]}
        self.kind = {n["id"]: n["kind"] for n in ir["nodes"]}
        self.fwd = defaultdict(set)     # caller -> callees
        self.rev = defaultdict(set)     # callee -> callers
        self.radj = defaultdict(list)   # PPR propagation: callee -> [callers]
        self.extern_out = defaultdict(int)
        self.resolved_out = defaultdict(int)
        self.by_name = defaultdict(list)
        nodes = set(self.meta)
        for n in ir["nodes"]:
            if n["kind"] in ("function", "method"):
                self.by_name[n["name"]].append(n["id"])
        for e in ir["edges"]:
            if e["rel"] != "calls":
                continue
            s, d = e["src"], e["dst"]
            self.fwd[s].add(d); self.rev[d].add(s); self.radj[d].append(s)
            nodes.add(s); nodes.add(d)
            if self.kind.get(d) in ("extern", "ambiguous"):
                self.extern_out[s] += 1
            else:
                self.resolved_out[s] += 1
        self.nodes = sorted(nodes)
        self.out_edges = {k: sorted(set(v)) for k, v in self.radj.items()}
        self.taint = taint or {}
        t = self.taint
        self.sources, self.sinks, self.tsinks = (t.get("sources", {}),
                                                 t.get("sinks", {}), t.get("taint_sinks", {}))

    # ---- seeds ----
    def names_to_nodes(self, names):
        out = []
        for nm in names:
            out.extend(self.by_name.get(nm, []))
        return out

    def _reach(self, adj, start):
        seen, stack = set(), list(adj.get(start, ()))
        while stack:
            n = stack.pop()
            if n in seen:
                continue
            seen.add(n); stack.extend(adj.get(n, ()))
        seen.discard(start)
        return seen

    def _ppr(self, pvec):
        n = len(self.nodes)
        r = {nid: pvec.get(nid, 0.0) for nid in self.nodes}
        if sum(r.values()) == 0:
            r = {nid: 1.0 / n for nid in self.nodes}
        for _ in range(ITERS):
            nxt = {nid: (1.0 - D) * pvec.get(nid, 0.0) for nid in self.nodes}
            dangling = 0.0
            for v in self.nodes:
                tv = self.out_edges.get(v, ())
                if not tv:
                    dangling += r[v]; continue
                share = D * r[v] / len(tv)
                for tt in tv:
                    nxt[tt] += share
            if dangling:
                for nid in self.nodes:
                    nxt[nid] += D * dangling * pvec.get(nid, 0.0)
            s = sum(nxt.values()) or 1.0
            r = {k: v / s for k, v in nxt.items()}
        return r

    # ---- scoring ----
    def score(self, seed: dict, top_n: int = 15) -> list:
        tot = sum(seed.values()) or 1.0
        pvec = {k: v / tot for k, v in seed.items()}
        ppr = self._ppr(pvec)
        rows = []
        for nid in (n for n in self.nodes if self.kind.get(n) in ("function", "method")):
            ro, eo = self.resolved_out.get(nid, 0), self.extern_out.get(nid, 0)
            m = self.meta[nid]
            rows.append({
                "id": nid, "name": m["name"], "file": m["file"], "line": m["line"],
                "fan_in": len(self.rev.get(nid, ())), "fan_out": len(self.fwd.get(nid, ())),
                "downstream": len(self._reach(self.fwd, nid)),
                "upstream": len(self._reach(self.rev, nid)),
                "coherence": round((ro / (ro + eo)) if (ro + eo) else 0.5, 4),
                "is_sink": nid in self.sinks, "taint_sink": nid in self.tsinks,
                "taint_patterns": self.sinks.get(nid, []) or self.sources.get(nid, []),
                "ppr": ppr.get(nid, 0.0),
            })
        tp = sum(x["ppr"] for x in rows) or 1.0
        for x in rows:
            x["prob"] = round(x["ppr"] / tp, 6); del x["ppr"]
        rows.sort(key=lambda x: (-x["prob"], x["id"]))
        for i, x in enumerate(rows):
            x["rank"] = i + 1
        return rows[:top_n]

    def score_by_names(self, names, top_n: int = 15, blend_taint: bool = True) -> list:
        """Seed from traceback/issue function NAMES; optionally blend the taint
        seed so the walk still favors taint-reachable sinks near the frame."""
        seed = defaultdict(float)
        for nid in self.names_to_nodes(names):
            seed[nid] += 1.0
        if blend_taint:
            for nid, w in self.taint.get("seed", {}).items():
                seed[nid] += 0.25 * w
        if not seed:  # unknown name -> fall back to taint-only
            seed = dict(self.taint.get("seed", {}))
        return self.score(dict(seed), top_n)


if __name__ == "__main__":
    import sys, json, cache
    cid = sys.argv[1]
    names = [a for a in sys.argv[2:] if not a.isdigit()]
    top_n = next((int(a) for a in sys.argv[2:] if a.isdigit()), 10)
    ir = cache.read(cid, "semdag_ir.json")
    try:
        taint = cache.read(cid, "taint.json")
    except FileNotFoundError:
        taint = None
    scorer = PurePythonPPRScorer(ir, taint)
    hubs = scorer.score_by_names(names, top_n)
    print(json.dumps({"seed_names": names, "candidates_ranked": len(hubs),
        "top": [{k: h[k] for k in ("rank","name","file","line","prob","is_sink",
                 "taint_sink","downstream","taint_patterns")} for h in hubs]}, indent=2))


In [ ]:
%%writefile /kaggle/working/repro/legacy_semdag/repo_eval.py
"""repo_eval — deterministic repo-state automaton (the external verifier).

Turns a whole repo into ONE compact eval.json the agent reads instead of
spinning sub-agents. Stdlib-only, offline, reproducible. The agent calls this
via run_command, reads the summary, edits, and calls it again to verify — the
compiler + the repo's own tests are the non-driftable oracle.

Stages (cheap/deterministic by default; build+test are opt-in to protect the
per-task compute budget):

  1. snapshot        — freeze repo bytes (snapshot.json: per-file sha256 + tree hash)
  2. semIR DAG       — parse to nodes/edges (stdlib `ast`)
  3. PARITY CHECK    — prove the DAG is a faithful, non-drifted derivation of the
                       frozen snapshot: recompute the tree hash from snapshot
                       files, require it == snapshot.tree_sha256 == ir.tree_sha256,
                       and require every IR file-node to exist in the snapshot
                       (no phantom/drifted files). Hard gate: parity.ok.
  4. localization    — PPR seeded by symbols lifted from the problem statement →
                       ranked candidate edit sites.
  5. syntax          — py_compile every touched .py (always available).
  6. typecheck       — mypy/pyright on changed files IF present (else skipped).
  7. build  (--build "<cmd>")  — run it; report rc + log tail.
  8. tests  (--test  "<cmd>")  — run the repo's OWN tests; report pass/fail +
                       failing nodeids/file:line/trace. NEVER the hidden test_patch.

Usage:
  python repo_eval.py <container_id> <repo_dir> [--problem-file F | --problem "..."]
      [--changed f1.py,f2.py] [--typecheck] [--build "pip install -e ."]
      [--test "python -m pytest -q"] [--top-n N]
"""
import os, sys, re, json, hashlib, subprocess, argparse
import cache, snapshot, semdag_ir_ast, taint
from ppr_scorer import PurePythonPPRScorer

GUARDRAIL = "repo_eval never reads held-out/hidden tests (test_patch); test signal uses the repo's own tests only."

def _recompute_tree(snap: dict) -> str:
    h = hashlib.sha256()
    for rel in sorted(snap["files"]):
        h.update(rel.encode()); h.update(snap["files"][rel]["sha256"].encode())
    return h.hexdigest()

def parity_check(container_id: str, snap: dict, ir: dict) -> dict:
    recomputed = _recompute_tree(snap)
    snap_tree = snap["tree_sha256"]; ir_tree = ir["tree_sha256"]
    ir_files = {n["file"] for n in ir["nodes"] if n["kind"] == "file"}
    phantom = sorted(f for f in ir_files if f not in snap["files"])  # drift: IR file not in snapshot
    py_in_snap = {r for r in snap["files"] if r.endswith(".py") and "content" in snap["files"][r]}
    covered = sorted(ir_files & py_in_snap)
    uncovered = sorted(py_in_snap - ir_files)  # parseable files the IR missed
    ok = (recomputed == snap_tree == ir_tree) and not phantom
    return {"ok": ok, "tree_sha256": snap_tree,
            "recomputed_matches_snapshot": recomputed == snap_tree,
            "ir_matches_snapshot": ir_tree == snap_tree,
            "phantom_files": phantom, "py_files": len(py_in_snap),
            "covered": len(covered), "uncovered_parseable": uncovered[:20]}

_SYM = re.compile(r"[A-Za-z_][A-Za-z0-9_]{2,}")
def problem_symbols(text: str):
    if not text:
        return []
    # prefer backtick/quoted tokens, then dotted.attrs, then bare identifiers
    toks = re.findall(r"`([^`]+)`|'([^']+)'|\"([^\"]+)\"", text)
    flat = [t for grp in toks for t in grp if t]
    for m in re.findall(r"[A-Za-z_][\w\.]+", text):
        flat.append(m.split(".")[-1])
    seen, out = set(), []
    for t in flat:
        for w in _SYM.findall(t):
            if w not in seen:
                seen.add(w); out.append(w)
    return out[:40]

def py_compile_errors(snap: dict, changed=None):
    import ast as _ast
    errs = []
    targets = changed or [r for r in snap["files"] if r.endswith(".py") and "content" in snap["files"][r]]
    for rel in targets:
        e = snap["files"].get(rel, {})
        if "content" not in e:
            continue
        try:
            _ast.parse(e["content"], filename=rel)
        except SyntaxError as ex:
            errs.append({"file": rel, "line": ex.lineno, "msg": str(ex.msg)})
    return errs

def typecheck(repo: str, changed):
    import shutil
    tool = next((t for t in ("mypy", "pyright") if shutil.which(t)), None)
    if not tool or not changed:
        return {"ran": False, "tool": tool, "errors": []}
    try:
        p = subprocess.run([tool, *changed], cwd=repo, capture_output=True, text=True, timeout=120)
        errs = [l for l in p.stdout.splitlines() if ": error" in l][:50]
        return {"ran": True, "tool": tool, "errors": errs}
    except Exception as ex:
        return {"ran": False, "tool": tool, "error": str(ex)}

_PYTEST_FAIL = re.compile(r"^(FAILED|ERROR)\s+(?P<node>\S+)", re.M)
def run_cmd(repo, cmd, timeout):
    try:
        p = subprocess.run(cmd, cwd=repo, shell=True, capture_output=True, text=True, timeout=timeout)
        return p.returncode, (p.stdout + "\n" + p.stderr)
    except Exception as ex:
        return 1, f"runner error: {ex}"

def verify_delta(container_id: str, repo: str, test_cmd=None):
    """Persistent incremental verify: diff the working tree against the FROZEN
    baseline snapshot (set on the first full run), re-check only changed files,
    and return a clean/regressed verdict. Cheap enough to run after every edit."""
    base = cache.read(container_id, "snapshot.json")   # the frozen baseline
    bfiles = base["files"]
    changed, added, removed = [], [], []
    seen = set()
    for root, dirs, names in os.walk(repo):
        dirs[:] = [d for d in dirs if d not in snapshot.SKIP_DIRS]
        for name in sorted(names):
            fp = os.path.join(root, name)
            if not os.path.isfile(fp):
                continue
            rel = os.path.relpath(fp, repo); seen.add(rel)
            try:
                sha = cache.sha256_file(fp)
            except OSError:
                continue
            if rel not in bfiles:
                added.append(rel)
            elif bfiles[rel]["sha256"] != sha:
                changed.append(rel)
    removed = [r for r in bfiles if r not in seen]
    touched = sorted(set(changed + added))
    # re-parse + syntax only the touched .py (incremental; baseline stays frozen)
    import ast as _ast
    syntax = []
    for rel in touched:
        if not rel.endswith(".py"):
            continue
        fp = os.path.join(repo, rel)
        try:
            _ast.parse(open(fp, errors="replace").read(), filename=rel)
        except SyntaxError as ex:
            syntax.append({"file": rel, "line": ex.lineno, "msg": str(ex.msg)})
    tests = None
    if test_cmd:
        rc, log = run_cmd(repo, test_cmd, 600)
        fails = [{"node": m.group("node")} for m in _PYTEST_FAIL.finditer(log)][:50]
        tests = {"cmd": test_cmd, "rc": rc, "passed": rc == 0, "failures": fails,
                 "log_tail": log.strip().splitlines()[-12:]}
    clean = (not syntax) and (tests is None or tests["passed"])
    verdict = {"mode": "verify", "guardrail": GUARDRAIL,
               "baseline_tree_sha256": base["tree_sha256"][:12],
               "changed": changed, "added": added, "removed": removed,
               "syntax_errors": syntax, "tests": tests, "clean": clean}
    cache.write(container_id, "verify.json", verdict)
    cache.mark_stage(container_id, "verify",
                     {"clean": clean, "touched": len(touched), "syntax_errors": len(syntax)})
    print(json.dumps({"clean": clean, "changed": changed, "added": added,
        "removed": removed, "syntax_errors": syntax[:5],
        "tests": (tests and {"passed": tests["passed"], "failures": len(tests["failures"])})},
        indent=2))
    return verdict

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("container_id"); ap.add_argument("repo_dir")
    ap.add_argument("--problem"); ap.add_argument("--problem-file")
    ap.add_argument("--changed"); ap.add_argument("--typecheck", action="store_true")
    ap.add_argument("--build"); ap.add_argument("--test")
    ap.add_argument("--top-n", type=int, default=10)
    ap.add_argument("--verify", action="store_true",
                    help="incremental verify vs the frozen baseline (run after each edit)")
    a = ap.parse_args()
    if a.verify:
        verify_delta(a.container_id, os.path.abspath(a.repo_dir), a.test)
        return
    repo = os.path.abspath(a.repo_dir)
    problem = a.problem or (open(a.problem_file).read() if a.problem_file else "")
    changed = [c.strip() for c in a.changed.split(",")] if a.changed else None

    cache.init_container(a.container_id, repo)
    snap = snapshot.snapshot(a.container_id, repo)
    ir = semdag_ir_ast.build(a.container_id)
    par = parity_check(a.container_id, snap, ir)

    # localization (only meaningful when parity holds — else the DAG is untrusted)
    localization = []
    if par["ok"]:
        tnt = taint.build(a.container_id)
        scorer = PurePythonPPRScorer(ir, tnt)
        syms = problem_symbols(problem)
        hubs = scorer.score_by_names(syms, a.top_n) if syms else scorer.score(tnt.get("seed", {}), a.top_n)
        localization = [{k: h[k] for k in ("rank", "name", "file", "line", "prob")} for h in hubs]

    syntax = py_compile_errors(snap, changed)
    types = typecheck(repo, changed) if a.typecheck else {"ran": False}
    build = None
    if a.build:
        rc, log = run_cmd(repo, a.build, 600)
        build = {"cmd": a.build, "rc": rc, "ok": rc == 0, "log_tail": log.strip().splitlines()[-15:]}
    tests = None
    if a.test:
        rc, log = run_cmd(repo, a.test, 600)
        fails = [{"node": m.group("node")} for m in _PYTEST_FAIL.finditer(log)][:50]
        tests = {"cmd": a.test, "rc": rc, "passed": rc == 0,
                 "failures": fails, "log_tail": log.strip().splitlines()[-15:]}

    out = {"guardrail": GUARDRAIL, "repo": repo, "parity": par,
           "semdag": ir["counts"], "problem_symbols": problem_symbols(problem)[:12],
           "localization": localization, "syntax_errors": syntax,
           "typecheck": types, "build": build, "tests": tests}
    cache.write(a.container_id, "eval.json", out)
    cache.mark_stage(a.container_id, "repo_eval",
                     {"parity_ok": par["ok"], "syntax_errors": len(syntax),
                      "localized": len(localization)})
    # agent-facing compact summary
    print(json.dumps({"parity_ok": par["ok"], "tree_sha256": par["tree_sha256"][:12],
        "files": par["py_files"], "covered": par["covered"], "phantom": par["phantom_files"],
        "semdag": ir["counts"], "top_edit_sites": localization[:6],
        "syntax_errors": syntax[:5],
        "build": (build and {"ok": build["ok"]}),
        "tests": (tests and {"passed": tests["passed"], "failures": len(tests["failures"])})},
        indent=2))

if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/repro/legacy_semdag/semdag_ir_ast.py
"""Stage 3 (stdlib) — SemDAG-IR via Python's built-in `ast`. No tree-sitter, no
network, no compile: usable in a fully offline sandbox. Produces the SAME
semdag_ir.json schema the rest of the pipeline consumes:

  nodes: file / class / function|method  (+ extern:/ambiguous:/module:)
  edges: contains / calls / imports

Reads file contents from the cached snapshot (deterministic, context-preserving)
and applies identical scope-qualified call resolution (same-file wins, unique
global, self.-method -> same-class, else one ambiguous sink; extern if no def).

Covers Python targets. For multi-language repos where tree-sitter IS available
(pre-vendored), semdag_ir_flow.py remains the richer path; this is the
zero-dependency default.

Usage: python semdag_ir_ast.py <container_id>
"""
import os, sys, json, ast
import cache

def build(container_id: str) -> dict:
    snap = cache.read(container_id, "snapshot.json")
    nodes = {}
    edges = []
    defs_by_name = {}
    raw_calls = []  # (caller_id, name, rel, is_method, receiver)

    def add_node(nid, kind, name, file, line, cls=None):
        if nid not in nodes:
            n = {"id": nid, "kind": kind, "name": name, "file": file, "line": line}
            if cls is not None:
                n["cls"] = cls
            nodes[nid] = n
            if kind in ("function", "method"):
                defs_by_name.setdefault(name, []).append(nid)

    for rel, entry in sorted(snap["files"].items()):
        if "content" not in entry or not rel.endswith(".py"):
            continue
        try:
            tree = ast.parse(entry["content"])
        except SyntaxError:
            continue
        file_id = f"file:{rel}"
        add_node(file_id, "file", rel, rel, 1)

        def callee(node):
            """(name, is_method, receiver) for an ast.Call."""
            f = node.func
            if isinstance(f, ast.Attribute):
                recv = f.value.id if isinstance(f.value, ast.Name) else None
                return f.attr, True, recv
            if isinstance(f, ast.Name):
                return f.id, False, None
            # e.g. chained call()(); dig for the last attribute/name
            cur = f
            while isinstance(cur, (ast.Call, ast.Subscript)):
                cur = cur.func if isinstance(cur, ast.Call) else cur.value
            if isinstance(cur, ast.Attribute):
                return cur.attr, True, None
            if isinstance(cur, ast.Name):
                return cur.id, False, None
            return None, False, None

        def walk(node, scope_id, scope_cls):
            for child in ast.iter_child_nodes(node):
                if isinstance(child, ast.ClassDef):
                    cid = f"{rel}::{child.name}#cls@{child.lineno}"
                    add_node(cid, "class", child.name, rel, child.lineno)
                    edges.append({"src": scope_id, "dst": cid, "rel": "contains"})
                    walk(child, cid, child.name)
                elif isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)):
                    kind = "method" if scope_cls is not None else "function"
                    fid = f"{rel}::{child.name}#fn@{child.lineno}"
                    add_node(fid, kind, child.name, rel, child.lineno, cls=scope_cls)
                    edges.append({"src": scope_id, "dst": fid, "rel": "contains"})
                    walk(child, fid, None)  # nested defs/calls attributed to this fn
                elif isinstance(child, (ast.Attribute, ast.Subscript, ast.Name)):
                    _ids = [n.id for n in ast.walk(child) if isinstance(n, ast.Name)]
                    for _src in ("request", "argv", "environ", "stdin", "params", "form", "body"):
                        if _src in _ids and scope_id != file_id:
                            raw_calls.append((scope_id, "__source_" + _src, rel, False, None))
                    walk(child, scope_id, scope_cls)
                elif isinstance(child, ast.Call):
                    nm, is_m, recv = callee(child)
                    if nm:
                        raw_calls.append((scope_id, nm, rel, is_m, recv))
                    walk(child, scope_id, scope_cls)
                elif isinstance(child, ast.Import):
                    for a in child.names:
                        mid = f"module:{a.name}"
                        add_node(mid, "module", a.name, rel, child.lineno)
                        edges.append({"src": file_id, "dst": mid, "rel": "imports"})
                elif isinstance(child, ast.ImportFrom):
                    mod = child.module or "."
                    mid = f"module:{mod}"
                    add_node(mid, "module", mod, rel, child.lineno)
                    edges.append({"src": file_id, "dst": mid, "rel": "imports"})
                    walk(child, scope_id, scope_cls)
                else:
                    walk(child, scope_id, scope_cls)

        walk(tree, file_id, None)

    # scope/type-qualified call resolution (identical policy to the TS front end)
    def meta_of(nid): return nodes.get(nid, {})
    for caller, name, caller_rel, is_method, receiver in raw_calls:
        cands = defs_by_name.get(name, [])
        chosen = None
        if not cands:
            dst = f"extern:{name}"; add_node(dst, "extern", name, "", 0); chosen = [dst]
        else:
            same = [c for c in cands if meta_of(c).get("file") == caller_rel]
            if len(same) == 1:
                chosen = same
            elif len(cands) == 1:
                chosen = cands
            elif is_method and receiver in ("self", "cls"):
                cc = meta_of(caller).get("cls")
                inclass = [c for c in cands
                           if meta_of(c).get("cls") == cc and meta_of(c).get("file") == caller_rel]
                chosen = inclass if len(inclass) == 1 else None
            if chosen is None:
                dst = f"ambiguous:{name}"; add_node(dst, "ambiguous", name, "", 0); chosen = [dst]
        for t in chosen:
            if t != caller:
                edges.append({"src": caller, "dst": t, "rel": "calls"})

    uniq = sorted({(e["src"], e["dst"], e["rel"]) for e in edges})
    ir = {
        "tree_sha256": snap["tree_sha256"],
        "front_end": "stdlib-ast",
        "nodes": [nodes[k] for k in sorted(nodes)],
        "edges": [{"src": s, "dst": d, "rel": r} for s, d, r in uniq],
        "counts": {"nodes": len(nodes), "edges": len(uniq),
                   "calls": sum(1 for _, _, r in uniq if r == "calls")},
    }
    cache.write(container_id, "semdag_ir.json", ir)
    cache.mark_stage(container_id, "semdag_ir", ir["counts"])
    return ir

if __name__ == "__main__":
    cid = sys.argv[1]
    ir = build(cid)
    print(json.dumps({"stage": "semdag_ir(ast)", **ir["counts"]}))


In [ ]:
%%writefile /kaggle/working/repro/legacy_semdag/snapshot.py
"""Stage 2 — exec snapshot: capture a rollback-preserving snapshot of the repo
into the SharedContainerCache. Deterministic: file inventory + per-file sha256
+ a single tree hash. Stores content for small text files so a rollback can
restore them without re-reading the working tree. Writes snapshot.json.

Usage: python snapshot.py <container_id> <target_dir>
"""
import os, sys, json, hashlib
import cache

SKIP_DIRS = {".git", "target", "node_modules", ".venv", "venv", "__pycache__",
             ".cache", "dist", "build", ".claude", ".idea", ".vscode",
             ".pytest_cache", ".ruff_cache", ".mypy_cache", ".tox"}
MAX_STORE_BYTES = 256 * 1024  # store contents only for small text files

def is_texty(fp: str) -> bool:
    try:
        with open(fp, "rb") as f:
            chunk = f.read(4096)
        return b"\x00" not in chunk
    except Exception:
        return False

def snapshot(container_id: str, target: str) -> dict:
    target = os.path.abspath(target)
    files = {}
    tree = hashlib.sha256()
    for root, dirs, names in os.walk(target):
        dirs[:] = sorted(d for d in dirs if d not in SKIP_DIRS)
        for name in sorted(names):
            fp = os.path.join(root, name)
            if not os.path.isfile(fp):  # isfile follows symlinks; os.walk won't loop on dir links
                continue
            rel = os.path.relpath(fp, target)
            try:
                size = os.path.getsize(fp)
                digest = cache.sha256_file(fp)
            except OSError:
                continue
            entry = {"size": size, "sha256": digest}
            if size <= MAX_STORE_BYTES and is_texty(fp):
                with open(fp, "r", errors="replace") as f:
                    entry["content"] = f.read()
            files[rel] = entry
            tree.update(rel.encode()); tree.update(digest.encode())
    snap = {"target": target, "file_count": len(files),
            "tree_sha256": tree.hexdigest(), "files": files}
    cache.write(container_id, "snapshot.json", snap)
    cache.mark_stage(container_id, "snapshot",
                     {"files": len(files), "tree_sha256": snap["tree_sha256"]})
    return snap

def restore(container_id: str) -> int:
    """Rollback: rewrite stored file contents back to the target tree."""
    snap = cache.read(container_id, "snapshot.json")
    target = snap["target"]; n = 0
    for rel, e in snap["files"].items():
        if "content" in e:
            fp = os.path.join(target, rel)
            os.makedirs(os.path.dirname(fp), exist_ok=True)
            with open(fp, "w") as f:
                f.write(e["content"])
            n += 1
    return n

if __name__ == "__main__":
    cid, target = sys.argv[1], sys.argv[2]
    s = snapshot(cid, target)
    # compact line to stdout only (agent-facing); full data stays in cache
    print(json.dumps({"stage": "snapshot", "files": s["file_count"],
                      "tree_sha256": s["tree_sha256"]}))


In [ ]:
%%writefile /kaggle/working/repro/legacy_semdag/taint.py
"""Stage 3.5 — Taint seeding: a lightweight, deterministic data-flow overlay on
the SemDAG that marks where untrusted input enters (sources) and where danger
happens (sinks), then finds taint-sink intersections — sinks reachable from a
source over the call-flow DAG — and emits a Personalized PageRank seed vector.

This is node-granularity taint (not full def-use): a function is a SOURCE if it
or its outgoing calls match source patterns, a SINK if they match sink patterns.
A sink is TAINT-REACHABLE if some source can reach it via `calls` edges. Those
intersections get the heaviest seed mass, so the downstream PPR walk targets
SQLi / unsafe-deserialization / command-exec / memory-safety surface first.

Writes taint.json. Usage: python taint.py <container_id>
"""
import sys, json, re
from collections import defaultdict
import cache

# substring patterns (lowercased) matched against a node's name and the names of
# its outgoing extern/ambiguous callees + imported modules in its file.
SOURCE_PATTERNS = [
    "__source_",
    "request", "req.", "query", "params", "getquery", "formvalue", "body",
    "argv", "args", "clap", "getopt", "env::var", "getenv", "environ", "std::env",
    "stdin", "readline", "input(", "read_to_string", "fs::read", "readfile",
    "fopen", "open(", "recv", "socket", "accept", "deserialize", "from_str",
    "from_slice", "pickle.load", "json.load", "serde_json::from", "unmarshal",
]
SINK_PATTERNS = [
    # command / code execution
    "exec", "eval", "system(", "popen", "subprocess", "command::new", "spawn",
    "os.system", "child_process", "shell",
    # SQL
    "execute", "executemany", "query(", "raw_query", "prepare", "cursor.execute",
    "sqlx::query", "rusqlite", "format!(\"select", "\"select ", "\"insert ",
    "\"update ", "\"delete ",
    # deserialization (untrusted)
    "pickle.loads", "yaml.load", "marshal.loads", "deserialize", "from_reader",
    "serde_json::from", "bincode::deserialize",
    # memory safety / unsafe
    "unsafe", "transmute", "from_raw", "memcpy", "strcpy", "get_unchecked",
    "as_mut_ptr", "set_len", "slice::from_raw",
]

def classify(text: str, patterns) -> list:
    t = text.lower()
    return sorted({p for p in patterns if p in t})

def reachable(adj, start):
    seen, stack = set(), list(adj.get(start, ()))
    while stack:
        n = stack.pop()
        if n in seen:
            continue
        seen.add(n); stack.extend(adj.get(n, ()))
    seen.discard(start)
    return seen

def build(container_id: str) -> dict:
    ir = cache.read(container_id, "semdag_ir.json")
    meta = {n["id"]: n for n in ir["nodes"]}

    # per-function "signal text": its own name + names of callees (extern/ambiguous/internal)
    sig = defaultdict(list)
    fwd = defaultdict(set)   # caller -> callees (calls)
    imports_by_file = defaultdict(list)
    for n in ir["nodes"]:
        if n["kind"] == "module":
            imports_by_file[n["file"]].append(n["name"])
    for e in ir["edges"]:
        if e["rel"] == "calls":
            fwd[e["src"]].add(e["dst"])
            sig[e["src"]].append(meta.get(e["dst"], {}).get("name", ""))

    funcs = [n for n in ir["nodes"] if n["kind"] in ("function", "method")]
    sources, sinks = {}, {}
    for n in funcs:
        nid = n["id"]
        text = " ".join([n["name"]] + sig.get(nid, []) + imports_by_file.get(n["file"], []))
        sm = classify(text, SOURCE_PATTERNS)
        km = classify(text, SINK_PATTERNS)
        if sm:
            sources[nid] = sm
        if km:
            sinks[nid] = km

    # taint-sink intersection: a sink reachable from some source via calls flow
    src_reach = {}
    for s in sources:
        src_reach[s] = reachable(fwd, s)
    taint_sinks = {}
    for k in sinks:
        feeders = [s for s in sources if k in src_reach.get(s, ()) or k == s]
        if feeders:
            taint_sinks[k] = {"patterns": sinks[k], "fed_by": sorted(feeders)[:8]}

    # Personalized PageRank seed vector:
    #   taint-reachable sink = 1.0 (the bug gathers here)
    #   plain sink           = 0.5
    #   source               = 0.25 (entry surface)
    seed = defaultdict(float)
    for nid in sinks:
        seed[nid] += 1.0 if nid in taint_sinks else 0.5
    for nid in sources:
        seed[nid] += 0.25
    tot = sum(seed.values()) or 1.0
    seed = {k: round(v / tot, 8) for k, v in seed.items()}

    out = {
        "tree_sha256": ir["tree_sha256"],
        "counts": {"sources": len(sources), "sinks": len(sinks),
                   "taint_sinks": len(taint_sinks), "seed_nodes": len(seed)},
        "sources": sources, "sinks": sinks, "taint_sinks": taint_sinks,
        "seed": seed,
    }
    cache.write(container_id, "taint.json", out)
    cache.mark_stage(container_id, "taint", out["counts"])
    return out

if __name__ == "__main__":
    cid = sys.argv[1]
    o = build(cid)
    print(json.dumps({"stage": "taint", **o["counts"]}))


## Production SemDAG (the files the `v4_semdag` skill ships)

In [ ]:
%%writefile /kaggle/working/repro/semdag/cache.py
"""SharedContainerCache: one namespace per container ("one probe per container").
All pipeline stages read/write here; the orchestrating agent reads only the
compact results, never the repo text — context is preserved by construction.

Layout:  <root>/<container_id>/{snapshot.json, semdag_ir.json, vulcan_ranking.json, meta.json}
Default root: $SEMDAG_CACHE or ~/.cache/semdag
"""
import os, json, time, hashlib

def cache_root() -> str:
    return os.environ.get("SEMDAG_CACHE", os.path.expanduser("~/.cache/semdag"))

def container_dir(container_id: str) -> str:
    d = os.path.join(cache_root(), container_id)
    os.makedirs(d, exist_ok=True)
    return d

def init_container(container_id: str, target: str) -> dict:
    d = container_dir(container_id)
    meta = {"container_id": container_id, "target": os.path.abspath(target),
            "created": time.time(), "stages": {}}
    write(container_id, "meta.json", meta)
    return meta

def path(container_id: str, name: str) -> str:
    return os.path.join(container_dir(container_id), name)

def write(container_id: str, name: str, obj) -> str:
    p = path(container_id, name)
    tmp = p + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, sort_keys=True, separators=(",", ":"))
    os.replace(tmp, p)  # atomic
    return p

def read(container_id: str, name: str):
    with open(path(container_id, name)) as f:
        return json.load(f)

def mark_stage(container_id: str, stage: str, info: dict):
    meta = read(container_id, "meta.json")
    meta.setdefault("stages", {})[stage] = {"at": time.time(), **info}
    write(container_id, "meta.json", meta)

def sha256_file(fp: str) -> str:
    h = hashlib.sha256()
    with open(fp, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 16), b""):
            h.update(chunk)
    return h.hexdigest()


In [ ]:
%%writefile /kaggle/working/repro/semdag/repo_eval.py
"""repo_eval — deterministic repo-state automaton (the external verifier).

Turns a whole repo into ONE compact eval.json the agent reads instead of
spinning sub-agents. Stdlib-only, offline, reproducible. The agent calls this
via run_command, reads the summary, edits, and calls it again to verify — the
compiler + the repo's own tests are the non-driftable oracle.

Stages (cheap/deterministic by default; build+test are opt-in to protect the
per-task compute budget):

  1. snapshot        — freeze repo bytes (snapshot.json: per-file sha256 + tree hash)
  2. semIR DAG       — parse to nodes/edges (stdlib `ast`)
  3. PARITY CHECK    — prove the DAG is a faithful, non-drifted derivation of the
                       frozen snapshot: recompute the tree hash from snapshot
                       files, require it == snapshot.tree_sha256 == ir.tree_sha256,
                       and require every IR file-node to exist in the snapshot
                       (no phantom/drifted files). Hard gate: parity.ok.
  4. localization    — BM25 over live function source supplies restart mass to
                       a light typed walk over contains/calls/imports.
  5. syntax          — py_compile every touched .py (always available).
  6. typecheck       — mypy/pyright on changed files IF present (else skipped).
  7. build  (--build "<cmd>")  — run it; report rc + log tail.
  8. tests  (--test  "<cmd>")  — run the repo's OWN tests; report pass/fail +
                       failing nodeids/file:line/trace. NEVER the hidden test_patch.

Usage:
  python repo_eval.py <container_id> <repo_dir> [--problem-file F | --problem "..."]
      [--changed f1.py,f2.py] [--typecheck] [--build "pip install -e ."]
      [--test "python -m pytest -q"] [--top-n N]
"""
import os, sys, re, json, hashlib, subprocess, argparse
# Run as an ADK skill script: sibling modules live beside this file, and the
# cache must stay out of /workspace so it never enters the submitted diff.
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
os.environ.setdefault("SEMDAG_CACHE", "/tmp/semdag-cache")
import cache, snapshot, semdag_ir_ast, retriever

GUARDRAIL = "repo_eval never reads held-out/hidden tests (test_patch); test signal uses the repo's own tests only."

def _recompute_tree(snap: dict) -> str:
    h = hashlib.sha256()
    for rel in sorted(snap["files"]):
        h.update(rel.encode()); h.update(snap["files"][rel]["sha256"].encode())
    return h.hexdigest()

def parity_check(container_id: str, snap: dict, ir: dict) -> dict:
    recomputed = _recompute_tree(snap)
    snap_tree = snap["tree_sha256"]; ir_tree = ir["tree_sha256"]
    ir_files = {n["file"] for n in ir["nodes"] if n["kind"] == "file"}
    phantom = sorted(f for f in ir_files if f not in snap["files"])  # drift: IR file not in snapshot
    py_in_snap = {r for r in snap["files"] if r.endswith(".py") and "content" in snap["files"][r]}
    covered = sorted(ir_files & py_in_snap)
    uncovered = sorted(py_in_snap - ir_files)  # parseable files the IR missed
    ok = (recomputed == snap_tree == ir_tree) and not phantom
    return {"ok": ok, "tree_sha256": snap_tree,
            "recomputed_matches_snapshot": recomputed == snap_tree,
            "ir_matches_snapshot": ir_tree == snap_tree,
            "phantom_files": phantom, "py_files": len(py_in_snap),
            "covered": len(covered), "uncovered_parseable": uncovered[:20]}

_SYM = re.compile(r"[A-Za-z_][A-Za-z0-9_]{2,}")
def problem_symbols(text: str):
    if not text:
        return []
    # prefer backtick/quoted tokens, then dotted.attrs, then bare identifiers
    toks = re.findall(r"`([^`]+)`|'([^']+)'|\"([^\"]+)\"", text)
    flat = [t for grp in toks for t in grp if t]
    for m in re.findall(r"[A-Za-z_][\w\.]+", text):
        flat.append(m.split(".")[-1])
    seen, out = set(), []
    for t in flat:
        for w in _SYM.findall(t):
            if w not in seen:
                seen.add(w); out.append(w)
    return out[:40]

def py_compile_errors(snap: dict, changed=None):
    import ast as _ast
    errs = []
    targets = changed or [r for r in snap["files"] if r.endswith(".py") and "content" in snap["files"][r]]
    for rel in targets:
        e = snap["files"].get(rel, {})
        if "content" not in e:
            continue
        try:
            _ast.parse(e["content"], filename=rel)
        except SyntaxError as ex:
            errs.append({"file": rel, "line": ex.lineno, "msg": str(ex.msg)})
    return errs

def typecheck(repo: str, changed):
    import shutil
    tool = next((t for t in ("mypy", "pyright") if shutil.which(t)), None)
    if not tool or not changed:
        return {"ran": False, "tool": tool, "errors": []}
    try:
        p = subprocess.run([tool, *changed], cwd=repo, capture_output=True, text=True, timeout=120)
        errs = [l for l in p.stdout.splitlines() if ": error" in l][:50]
        return {"ran": True, "tool": tool, "errors": errs}
    except Exception as ex:
        return {"ran": False, "tool": tool, "error": str(ex)}

_PYTEST_FAIL = re.compile(r"^(FAILED|ERROR)\s+(?P<node>\S+)", re.M)
def run_cmd(repo, cmd, timeout):
    try:
        p = subprocess.run(cmd, cwd=repo, shell=True, capture_output=True, text=True, timeout=timeout)
        return p.returncode, (p.stdout + "\n" + p.stderr)
    except Exception as ex:
        return 1, f"runner error: {ex}"

def verify_delta(container_id: str, repo: str, test_cmd=None):
    """Persistent incremental verify: diff the working tree against the FROZEN
    baseline snapshot (set on the first full run), re-check only changed files,
    and return a clean/regressed verdict. Cheap enough to run after every edit."""
    base = cache.read(container_id, "snapshot.json")   # the frozen baseline
    bfiles = base["files"]
    changed, added, removed = [], [], []
    seen = set()
    for root, dirs, names in os.walk(repo):
        dirs[:] = [d for d in dirs if d not in snapshot.SKIP_DIRS]
        for name in sorted(names):
            fp = os.path.join(root, name)
            if not os.path.isfile(fp):
                continue
            rel = os.path.relpath(fp, repo); seen.add(rel)
            try:
                sha = cache.sha256_file(fp)
            except OSError:
                continue
            if rel not in bfiles:
                added.append(rel)
            elif bfiles[rel]["sha256"] != sha:
                changed.append(rel)
    removed = [r for r in bfiles if r not in seen]
    touched = sorted(set(changed + added))
    # re-parse + syntax only the touched .py (incremental; baseline stays frozen)
    import ast as _ast
    syntax = []
    for rel in touched:
        if not rel.endswith(".py"):
            continue
        fp = os.path.join(repo, rel)
        try:
            _ast.parse(open(fp, errors="replace").read(), filename=rel)
        except SyntaxError as ex:
            syntax.append({"file": rel, "line": ex.lineno, "msg": str(ex.msg)})
    tests = None
    if test_cmd:
        rc, log = run_cmd(repo, test_cmd, 600)
        fails = [{"node": m.group("node")} for m in _PYTEST_FAIL.finditer(log)][:50]
        tests = {"cmd": test_cmd, "rc": rc, "passed": rc == 0, "failures": fails,
                 "log_tail": log.strip().splitlines()[-12:]}
    clean = (not syntax) and (tests is None or tests["passed"])
    verdict = {"mode": "verify", "guardrail": GUARDRAIL,
               "baseline_tree_sha256": base["tree_sha256"][:12],
               "changed": changed, "added": added, "removed": removed,
               "syntax_errors": syntax, "tests": tests, "clean": clean}
    cache.write(container_id, "verify.json", verdict)
    cache.mark_stage(container_id, "verify",
                     {"clean": clean, "touched": len(touched), "syntax_errors": len(syntax)})
    print(json.dumps({"clean": clean, "changed": changed, "added": added,
        "removed": removed, "syntax_errors": syntax[:5],
        "tests": (tests and {"passed": tests["passed"], "failures": len(tests["failures"])})},
        indent=2))
    return verdict

def _normalize_argv(argv):
    """Accept the forms a small model actually emits through run_skill_script:
    one string holding the issue text, or one string holding a stringified list."""
    args = list(argv[1:])
    if len(args) == 1 and args[0].lstrip().startswith("["):
        try:
            import ast as _ast
            parsed = _ast.literal_eval(args[0])
            if isinstance(parsed, (list, tuple)):
                args = [str(x) for x in parsed]
        except (ValueError, SyntaxError):
            pass
    if len(args) == 1 and not args[0].startswith("--"):
        args = ["task", "/workspace", "--problem", args[0]]
    return [argv[0]] + args


def main():
    sys.argv = _normalize_argv(sys.argv)
    ap = argparse.ArgumentParser()
    ap.add_argument("container_id"); ap.add_argument("repo_dir")
    ap.add_argument("--problem"); ap.add_argument("--problem-file")
    ap.add_argument("--changed"); ap.add_argument("--typecheck", action="store_true")
    ap.add_argument("--build"); ap.add_argument("--test")
    ap.add_argument("--top-n", type=int, default=10)
    ap.add_argument("--verify", action="store_true",
                    help="incremental verify vs the frozen baseline (run after each edit)")
    a = ap.parse_args()
    if a.verify:
        verify_delta(a.container_id, os.path.abspath(a.repo_dir), a.test)
        return
    repo = os.path.abspath(a.repo_dir)
    problem = a.problem or (open(a.problem_file).read() if a.problem_file else "")
    changed = [c.strip() for c in a.changed.split(",")] if a.changed else None

    cache.init_container(a.container_id, repo)
    snap = snapshot.snapshot(a.container_id, repo)
    ir = semdag_ir_ast.build(a.container_id)
    par = parity_check(a.container_id, snap, ir)

    # Localization uses the issue against live function source, then performs a
    # light walk over the typed SemDAG.  Source evidence remains dominant.
    localization = []
    retrieval = None
    if par["ok"]:
        retrieval = retriever.rank(ir, snap, problem, a.top_n)
        localization = [
            {"rank": i + 1, **{k: site[k] for k in ("name", "qualname", "file", "line", "end_line",
                                                       "lexical_rank", "lexical_score", "structural_score")}}
            for i, site in enumerate(retrieval["top"])
        ]

    syntax = py_compile_errors(snap, changed)
    types = typecheck(repo, changed) if a.typecheck else {"ran": False}
    build = None
    if a.build:
        rc, log = run_cmd(repo, a.build, 600)
        build = {"cmd": a.build, "rc": rc, "ok": rc == 0, "log_tail": log.strip().splitlines()[-15:]}
    tests = None
    if a.test:
        rc, log = run_cmd(repo, a.test, 600)
        fails = [{"node": m.group("node")} for m in _PYTEST_FAIL.finditer(log)][:50]
        tests = {"cmd": a.test, "rc": rc, "passed": rc == 0,
                 "failures": fails, "log_tail": log.strip().splitlines()[-15:]}

    out = {"guardrail": GUARDRAIL, "repo": repo, "parity": par,
           "semdag": ir["counts"], "retrieval": retrieval,
           "localization": localization, "syntax_errors": syntax,
           "typecheck": types, "build": build, "tests": tests}
    cache.write(a.container_id, "eval.json", out)
    cache.mark_stage(a.container_id, "repo_eval",
                     {"parity_ok": par["ok"], "syntax_errors": len(syntax),
                      "localized": len(localization)})
    # agent-facing compact summary
    print(json.dumps({"parity_ok": par["ok"], "tree_sha256": par["tree_sha256"][:12],
        "files": par["py_files"], "covered": par["covered"], "phantom": par["phantom_files"],
        "semdag": ir["counts"],
        "localizer": (retrieval and {k: retrieval[k] for k in ("method", "damping", "candidates",
                                                                 "matched_candidates", "relations_used")}),
        "top_edit_sites": localization[:6],
        "syntax_errors": syntax[:5],
        "build": (build and {"ok": build["ok"]}),
        "tests": (tests and {"passed": tests["passed"], "failures": len(tests["failures"])})},
        indent=2))

if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/repro/semdag/retriever.py
"""Lexical retrieval followed by a light walk over the typed SemDAG IR.

The issue queries live function source with BM25.  Those scores become the
restart distribution for a deterministic walk over resolved calls, lexical
containment, and repository imports.  A small damping value keeps source
evidence primary while allowing structure to consolidate file-level context.
"""
from collections import Counter, defaultdict
import math
import re

IDENT = re.compile(r"[A-Za-z_][A-Za-z0-9_]*")
CAMEL = re.compile(r"[A-Z]+(?=[A-Z][a-z])|[A-Z]?[a-z]+|[A-Z]+|\d+")
STOP = {"the", "and", "for", "that", "this", "with", "not", "but", "are", "was", "from",
        "have", "has", "when", "you", "can", "should", "would", "will", "use", "using",
        "def", "self", "return", "import", "none", "true", "false", "class", "if", "else",
        "in", "is", "of", "to", "it", "as", "on", "be", "or", "an", "at", "by", "we"}
DEFAULT_DAMPING = 0.15
DEFAULT_RELATION_WEIGHTS = {"contains": 1.0, "calls": 1.0, "imports": 1.0}


def is_test_path(rel):
    parts = rel.split("/"); name = parts[-1]
    return (any(p in ("tests", "test", "testing") for p in parts[:-1])
            or name.startswith("test_") or name.endswith("_test.py") or name == "conftest.py")


def is_tool_path(rel):
    """Submission/harness implementation files are never task edit sites."""
    name = rel.split("/")[-1]
    return rel.startswith("semdag/") or name.startswith(".adk_exec_")


def is_candidate_path(rel, include_tests=False):
    return not is_tool_path(rel) and (include_tests or not is_test_path(rel))


def tokens(text):
    out = []
    for ident in IDENT.findall(text):
        low = ident.lower()
        if len(low) >= 2 and low not in STOP:
            out.append(low)
        parts = [p.lower() for chunk in ident.split("_") for p in CAMEL.findall(chunk)]
        if len(parts) > 1:
            out.extend(p for p in parts if len(p) >= 2 and p not in STOP)
    return out


def _documents(ir, snap, include_tests=False):
    docs = []
    for node in ir["nodes"]:
        if node["kind"] not in ("function", "method") or not is_candidate_path(node["file"], include_tests):
            continue
        content = snap["files"].get(node["file"], {}).get("content")
        if content is None or not node.get("end_line"):
            continue
        lines = content.splitlines()
        start = node.get("start_line", node["line"])
        body = "\n".join(lines[start - 1:node["end_line"]])
        head = f"{node.get('qualname', node['name']).replace('.', ' ')} {node['file'].replace('/', ' ').replace('.py', '')}"
        terms = tokens(body) + tokens(head) * 3
        docs.append({"node": node, "terms": Counter(terms), "length": len(terms)})
    return docs


def _bm25(docs, query):
    n = len(docs); df = Counter()
    for doc in docs: df.update(doc["terms"].keys())
    avg = sum(d["length"] for d in docs) / max(1, n)
    scores = [0.0] * n
    for term, qtf in Counter(tokens(query)).items():
        freq = df.get(term)
        if not freq: continue
        weight = math.log(1 + (n - freq + 0.5) / (freq + 0.5)) * min(qtf, 3)
        for i, doc in enumerate(docs):
            tf = doc["terms"].get(term)
            if tf:
                scores[i] += weight * tf * 2.2 / (tf + 1.2 * (0.25 + 0.75 * doc["length"] / avg))
    return scores


def _walk(ir, restart, damping, weights, include_tests=False, direction="undirected"):
    keep = {n["id"] for n in ir["nodes"] if n["kind"] in ("file", "class", "function", "method")
            and is_candidate_path(n.get("file", ""), include_tests)}
    adj = defaultdict(lambda: defaultdict(float)); used = Counter()
    for edge in ir["edges"]:
        rel = edge["rel"]; weight = weights.get(rel, 0.0)
        if weight <= 0 or edge["src"] not in keep or edge["dst"] not in keep: continue
        if direction == "reverse_calls" and rel == "calls":
            adj[edge["dst"]][edge["src"]] += weight
        else:
            adj[edge["src"]][edge["dst"]] += weight
            adj[edge["dst"]][edge["src"]] += weight
        used[rel] += 1
    rank = dict(restart)
    for _ in range(30):
        nxt = defaultdict(float); lost = 0.0
        for node, mass in rank.items():
            neighbors = adj.get(node); total = sum(neighbors.values()) if neighbors else 0.0
            if not total:
                lost += damping * mass; continue
            for target, weight in neighbors.items():
                nxt[target] += damping * mass * weight / total
        for node, mass in restart.items():
            nxt[node] += (1.0 - damping + lost) * mass
        rank = nxt
    return rank, used


def rank(ir, snap, issue, top_n=10, damping=DEFAULT_DAMPING, include_tests=False,
         relation_weights=None, direction="undirected"):
    """Return ranked function sites plus auditable lexical/structural evidence."""
    docs = _documents(ir, snap, include_tests=include_tests)
    scores = _bm25(docs, issue)
    positive = [i for i, score in enumerate(scores) if score > 0]
    total = sum(scores[i] for i in positive) or 1.0
    restart = {docs[i]["node"]["id"]: scores[i] / total for i in positive}
    weights = dict(DEFAULT_RELATION_WEIGHTS if relation_weights is None else relation_weights)
    graph, used = _walk(ir, restart, damping, weights, include_tests, direction) if restart and damping else (restart, Counter())
    lexical_order = sorted(positive, key=lambda i: (-scores[i], docs[i]["node"]["file"], docs[i]["node"]["line"]))
    lexical_rank = {i: pos + 1 for pos, i in enumerate(lexical_order)}
    order = sorted(positive, key=lambda i: (-graph.get(docs[i]["node"]["id"], 0.0),
                                            docs[i]["node"]["file"], docs[i]["node"]["line"]))
    top = []
    for i in order[:top_n]:
        node = docs[i]["node"]
        top.append({"id": node["id"], "name": node["name"], "qualname": node.get("qualname", node["name"]),
                    "file": node["file"], "line": node["line"], "end_line": node["end_line"],
                    "lexical_rank": lexical_rank[i], "lexical_score": scores[i],
                    "structural_score": graph.get(node["id"], 0.0)})
    return {"method": "bm25_typed_semdag", "damping": damping, "direction": direction,
            "candidates": len(docs),
            "matched_candidates": len(positive), "relations_used": dict(used), "top": top}


In [ ]:
%%writefile /kaggle/working/repro/semdag/semdag_ir_ast.py
"""Build the deterministic, stdlib-only SemDAG structural IR.

The IR keeps lexical containment, conservative resolved calls, and imports
resolved to repository files when possible.  It is a structural graph, not a
claim of full Python semantics: dynamic dispatch and runtime imports remain
explicitly unresolved.
"""
import sys, json, ast
from collections import defaultdict
import cache

def build(container_id: str) -> dict:
    snap = cache.read(container_id, "snapshot.json")
    nodes = {}
    edges = []
    defs_by_name = defaultdict(list)
    defs_by_module = defaultdict(dict)
    defs_by_class = defaultdict(dict)
    raw_calls = []
    imports = defaultdict(lambda: {"modules": {}, "symbols": {}})
    module_by_file = {}
    file_by_module = {}

    def module_name(rel):
        name = rel[:-3].replace("/", ".")
        return name[:-9] if name.endswith(".__init__") else name

    def add_node(nid, kind, name, file, line, **extra):
        if nid not in nodes:
            n = {"id": nid, "kind": kind, "name": name, "file": file, "line": line}
            n.update({k: v for k, v in extra.items() if v is not None})
            nodes[nid] = n
            if kind in ("function", "method"):
                defs_by_name[name].append(nid)

    def dotted_expr(node):
        parts = []
        while isinstance(node, ast.Attribute):
            parts.append(node.attr); node = node.value
        if isinstance(node, ast.Name):
            parts.append(node.id)
            return ".".join(reversed(parts))
        return None

    py_entries = [(rel, entry) for rel, entry in sorted(snap["files"].items())
                  if "content" in entry and rel.endswith(".py")]
    for rel, _ in py_entries:
        mod = module_name(rel)
        module_by_file[rel] = mod
        file_by_module[mod] = rel

    for rel, entry in py_entries:
        try:
            tree = ast.parse(entry["content"])
        except SyntaxError:
            continue
        file_id = f"file:{rel}"
        mod = module_by_file[rel]
        add_node(file_id, "file", rel, rel, 1, module=mod)

        class Builder(ast.NodeVisitor):
            def __init__(self):
                self.stack = [(file_id, "file", None, [])]

            def parent(self): return self.stack[-1]

            def visit_ClassDef(self, node):
                parent_id, _, _, qual = self.parent()
                q = qual + [node.name]
                cid = f"{rel}::{'.'.join(q)}#cls@{node.lineno}"
                add_node(cid, "class", node.name, rel, node.lineno,
                         end_line=node.end_lineno, qualname=".".join(q), module=mod)
                edges.append({"src": parent_id, "dst": cid, "rel": "contains"})
                self.stack.append((cid, "class", node.name, q)); self.generic_visit(node); self.stack.pop()

            def _function(self, node):
                parent_id, parent_kind, parent_cls, qual = self.parent()
                q = qual + [node.name]
                kind = "method" if parent_kind == "class" else "function"
                fid = f"{rel}::{'.'.join(q)}#fn@{node.lineno}"
                start = min([node.lineno] + [d.lineno for d in node.decorator_list])
                add_node(fid, kind, node.name, rel, node.lineno, end_line=node.end_lineno,
                         start_line=start, qualname=".".join(q), cls=parent_cls,
                         module=mod, is_async=isinstance(node, ast.AsyncFunctionDef))
                edges.append({"src": parent_id, "dst": fid, "rel": "contains"})
                defs_by_module[mod].setdefault(node.name, []).append(fid)
                if parent_kind == "class":
                    defs_by_class[(mod, parent_cls)].setdefault(node.name, []).append(fid)
                self.stack.append((fid, "function", parent_cls, q)); self.generic_visit(node); self.stack.pop()

            visit_FunctionDef = _function
            visit_AsyncFunctionDef = _function

            def visit_Call(self, node):
                caller, kind, cls, _ = self.parent()
                if kind == "function":
                    target = dotted_expr(node.func)
                    if target:
                        raw_calls.append({"caller": caller, "target": target,
                                          "file": rel, "module": mod, "class": cls,
                                          "line": node.lineno})
                self.generic_visit(node)

            def visit_Import(self, node):
                for alias in node.names:
                    local = alias.asname or alias.name.split(".")[0]
                    imports[rel]["modules"][local] = alias.name
                    self._import_edge(alias.name, node.lineno)

            def visit_ImportFrom(self, node):
                level = node.level or 0
                package = mod.split(".") if rel.endswith("/__init__.py") or rel == "__init__.py" else mod.split(".")[:-1]
                prefix = package[:max(0, len(package) - level + 1)] if level else []
                target_mod = ".".join(prefix + ([node.module] if node.module else []))
                for alias in node.names:
                    if alias.name == "*": continue
                    imports[rel]["symbols"][alias.asname or alias.name] = (target_mod, alias.name)
                self._import_edge(target_mod, node.lineno)

            def _import_edge(self, target_mod, line):
                if not target_mod: return
                target_file = file_by_module.get(target_mod)
                if target_file:
                    dst = f"file:{target_file}"
                else:
                    dst = f"module:{target_mod}"
                    add_node(dst, "module", target_mod, "", line)
                edges.append({"src": file_id, "dst": dst, "rel": "imports"})

        Builder().visit(tree)

    def single(values): return values[0] if len(values) == 1 else None
    for call in raw_calls:
        target = call["target"]; parts = target.split("."); rel = call["file"]
        mod = call["module"]; cls = call["class"]; chosen = None; resolution = None
        if len(parts) == 1:
            imported = imports[rel]["symbols"].get(parts[0])
            if imported:
                chosen = single(defs_by_module.get(imported[0], {}).get(imported[1], [])); resolution = "from_import"
            if chosen is None:
                chosen = single(defs_by_module.get(mod, {}).get(parts[0], [])); resolution = "same_module"
        elif parts[0] in ("self", "cls") and cls:
            chosen = single(defs_by_class.get((mod, cls), {}).get(parts[-1], [])); resolution = "same_class"
        elif parts[0] in imports[rel]["modules"]:
            chosen = single(defs_by_module.get(imports[rel]["modules"][parts[0]], {}).get(parts[-1], [])); resolution = "module_import"
        else:
            chosen = single(defs_by_class.get((mod, parts[-2]), {}).get(parts[-1], [])); resolution = "class_qualified"
        if chosen is None and len(parts) == 1:
            chosen = single(defs_by_name.get(parts[0], [])); resolution = "unique_global"
        if chosen and chosen != call["caller"]:
            edges.append({"src": call["caller"], "dst": chosen, "rel": "calls",
                          "line": call["line"], "resolution": resolution})

    uniq = {}
    for edge in edges:
        key = (edge["src"], edge["dst"], edge["rel"])
        uniq.setdefault(key, edge)
    ir = {
        "tree_sha256": snap["tree_sha256"],
        "front_end": "stdlib-ast",
        "nodes": [nodes[k] for k in sorted(nodes)],
        "edges": [uniq[k] for k in sorted(uniq)],
        "counts": {"nodes": len(nodes), "edges": len(uniq),
                   "calls": sum(1 for _, _, r in uniq if r == "calls"),
                   "contains": sum(1 for _, _, r in uniq if r == "contains"),
                   "imports": sum(1 for _, _, r in uniq if r == "imports")},
    }
    cache.write(container_id, "semdag_ir.json", ir)
    cache.mark_stage(container_id, "semdag_ir", ir["counts"])
    return ir

if __name__ == "__main__":
    cid = sys.argv[1]
    ir = build(cid)
    print(json.dumps({"stage": "semdag_ir(ast)", **ir["counts"]}))


In [ ]:
%%writefile /kaggle/working/repro/semdag/snapshot.py
"""Stage 2 — exec snapshot: capture a rollback-preserving snapshot of the repo
into the SharedContainerCache. Deterministic: file inventory + per-file sha256
+ a single tree hash. Stores content for small text files so a rollback can
restore them without re-reading the working tree. Writes snapshot.json.

Usage: python snapshot.py <container_id> <target_dir>
"""
import os, sys, json, hashlib
import cache

SKIP_DIRS = {".git", "target", "node_modules", ".venv", "venv", "__pycache__",
             ".cache", "dist", "build", ".claude", ".idea", ".vscode",
             ".pytest_cache", ".ruff_cache", ".mypy_cache", ".tox"}
MAX_STORE_BYTES = 256 * 1024  # store contents only for small text files

def is_texty(fp: str) -> bool:
    try:
        with open(fp, "rb") as f:
            chunk = f.read(4096)
        return b"\x00" not in chunk
    except Exception:
        return False

def snapshot(container_id: str, target: str) -> dict:
    target = os.path.abspath(target)
    files = {}
    for root, dirs, names in os.walk(target):
        dirs[:] = sorted(d for d in dirs if d not in SKIP_DIRS)
        for name in sorted(names):
            fp = os.path.join(root, name)
            if not os.path.isfile(fp):  # isfile follows symlinks; os.walk won't loop on dir links
                continue
            rel = os.path.relpath(fp, target)
            try:
                size = os.path.getsize(fp)
                digest = cache.sha256_file(fp)
            except OSError:
                continue
            entry = {"size": size, "sha256": digest}
            if size <= MAX_STORE_BYTES and is_texty(fp):
                with open(fp, "r", errors="replace") as f:
                    entry["content"] = f.read()
            files[rel] = entry
    # Hash the completed inventory in one canonical order.  os.walk ordering is
    # deterministic per directory, but it is not the same as a global path
    # sort when a directory and a sibling file share a prefix.  parity_check()
    # deliberately recomputes from sorted paths, so snapshot creation must use
    # that identical definition.
    tree = hashlib.sha256()
    for rel in sorted(files):
        tree.update(rel.encode()); tree.update(files[rel]["sha256"].encode())
    snap = {"target": target, "file_count": len(files),
            "tree_sha256": tree.hexdigest(), "files": files}
    cache.write(container_id, "snapshot.json", snap)
    cache.mark_stage(container_id, "snapshot",
                     {"files": len(files), "tree_sha256": snap["tree_sha256"]})
    return snap

def restore(container_id: str) -> int:
    """Rollback: rewrite stored file contents back to the target tree."""
    snap = cache.read(container_id, "snapshot.json")
    target = snap["target"]; n = 0
    for rel, e in snap["files"].items():
        if "content" in e:
            fp = os.path.join(target, rel)
            os.makedirs(os.path.dirname(fp), exist_ok=True)
            with open(fp, "w") as f:
                f.write(e["content"])
            n += 1
    return n

if __name__ == "__main__":
    cid, target = sys.argv[1], sys.argv[2]
    s = snapshot(cid, target)
    # compact line to stdout only (agent-facing); full data stays in cache
    print(json.dumps({"stage": "snapshot", "files": s["file_count"],
                      "tree_sha256": s["tree_sha256"]}))


## Measurement scripts

In [ ]:
%%writefile /kaggle/working/repro/phase1/measure.py
#!/usr/bin/env python3
"""Phase 1 measurement. Rules below are frozen. Do not retune after reading recall.

Question
    The class node is an outline (stub body). The method node is the real body.
    Parent id is recoverable by splitting the symbol id. Call edges do not encode
    that link. Does the sibling/child closure, or cosine after subtracting the
    parent-outline vector, cover the gold edit more often than the named body
    or one call hop?

Predictor inputs
    problem_statement, graph nodes/edges, 256-d node embeddings.
    The patch is used only to score. hints_text and test_patch are not read.

Gold alignment
    A .py hunk is anchored to the smallest node whose text contains the old-side
    line window (context lines and deletions, in order) as consecutive lines.
    If that window is absent, the consecutive deletion block is tried.
    A one-line anchor that occurs in more than AMBIGUOUS_LINE_CAP nodes is
    unaligned. Lines with strip length < MIN_LINE_CHARS do not count.
    New files and non-.py hunks are unaligned and sit only in the "all lines"
    denominator.

Arms (union over seeds; embeddings exclude the seed itself)
    Z     seed nodes named by the issue
    P     children of a class seed, else siblings under the same parent id
    ZP    Z union P
    C     callers and callees, one hop, uncapped
    C50   the same neighborhood truncated at 50 names in edge order
    E     top-k other nodes by raw cosine, k = |P| or FIXED_K if P is empty
    R     top-k by cosine after subtracting the parent outline (else the graph mean)
    RAND  mean of RANDOM_DRAWS uniform draws of size k, seeds excluded

Seeds
    Identifiers from the issue: backtick spans, dotted names, tokens before
    '(', snake_case, CamelCase. Drop length < 3, keywords, and any short name
    shared by more than AMBIGUITY_CAP nodes. Match exact id, then id suffix.

Selection
    Winner is the highest micro line-recall on non-Rich tasks among
    Z, P, ZP, C, E, R. Rich is confirmatory and is not an input to that choice.
    Contrasts are task-resampled on the same split: ZP-Z, R-E, R-C, Z-C.
"""

from __future__ import annotations

import hashlib
import json
import re
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np

ROOT = Path(__file__).resolve().parents[2]
DATA = ROOT / "DataSets" / "public" / "data"
TASKS = DATA / "tasks.jsonl"
GRAPHS = DATA / "graphs"
EMBEDS = DATA / "embeddings"
OUT = Path(__file__).resolve().parent / "out"

AMBIGUITY_CAP = 20
AMBIGUOUS_LINE_CAP = 5
FIXED_K = 10
NEIGHBOR_CAP = 50
RANDOM_DRAWS = 50
RNG_SEED = 0
MIN_LINE_CHARS = 3
N_BOOT = 4000
WINNER_ARMS = ("Z", "P", "ZP", "C", "E", "R")
PRIMARY_ARMS = ("Z", "P", "ZP", "C", "C50", "E", "R", "RAND")

KEYWORDS = {
    "and", "as", "assert", "async", "await", "break", "class", "continue",
    "def", "del", "elif", "else", "except", "false", "finally", "for", "from",
    "global", "if", "import", "in", "is", "lambda", "none", "nonlocal", "not",
    "or", "pass", "raise", "return", "true", "try", "while", "with", "yield",
}

IDENT_RE = re.compile(r"[A-Za-z_][A-Za-z0-9_]*")
DOTTED_RE = re.compile(r"[A-Za-z_][A-Za-z0-9_]*(?:\.[A-Za-z_][A-Za-z0-9_]*)+")
SNAKE_RE = re.compile(r"\b[a-z_][a-z0-9]*_[a-z0-9_]+\b")
CAMEL_RE = re.compile(r"\b[A-Z][a-zA-Z0-9]*[a-z][a-zA-Z0-9]*\b")
CALL_RE = re.compile(r"\b([A-Za-z_][A-Za-z0-9_]*)\s*\(")
TICK_RE = re.compile(r"`([^`\n]+)`")


def repo_short(repo: str) -> str:
    return repo.split("/")[-1]


def substantive(line: str) -> bool:
    return len(line.strip()) >= MIN_LINE_CHARS


def parse_hunks(patch: str) -> list[dict]:
    hunks: list[dict] = []
    old_path = None
    new_path = None
    buf: list[str] = []

    def flush() -> None:
        if not buf:
            return
        old_lines: list[str] = []
        deleted: list[str] = []
        added: list[str] = []
        for raw in buf:
            if raw.startswith("\\"):
                continue
            if raw.startswith("+"):
                added.append(raw[1:])
            elif raw.startswith("-"):
                deleted.append(raw[1:])
                old_lines.append(raw[1:])
            elif raw.startswith(" "):
                old_lines.append(raw[1:])
            else:
                old_lines.append(raw)
        path = new_path if old_path in (None, "/dev/null") else old_path
        hunks.append(
            {
                "path": path or "",
                "new_file": old_path in (None, "/dev/null"),
                "old_lines": old_lines,
                "deleted": deleted,
                "added": added,
            }
        )

    for line in patch.splitlines():
        if line.startswith("diff --git "):
            flush()
            buf = []
            old_path = None
            new_path = None
        elif line.startswith("--- "):
            old_path = line[4:].strip()
            if old_path.startswith("a/"):
                old_path = old_path[2:]
        elif line.startswith("+++ "):
            new_path = line[4:].strip()
            if new_path.startswith("b/"):
                new_path = new_path[2:]
        elif line.startswith("@@"):
            flush()
            buf = []
        elif old_path is not None or new_path is not None:
            if line.startswith(("+", "-", " ", "\\")):
                buf.append(line)
    flush()
    return hunks


def kind_of(text: str) -> str:
    for line in text.splitlines():
        stripped = line.lstrip()
        if not stripped or stripped.startswith("@"):
            continue
        if stripped.startswith("class "):
            return "class"
        return "func"
    return "func"


class Graph:
    def __init__(self, path: Path, embed_path: Path):
        raw = json.loads(path.read_text())
        self.ids: list[str] = []
        self.text: dict[str, str] = {}
        self.lines: dict[str, list[str]] = {}
        self.kind: dict[str, str] = {}
        self.nchars: dict[str, int] = {}
        for node in raw["nodes"]:
            ident = node["id"]
            body = node.get("text") or ""
            self.ids.append(ident)
            self.text[ident] = body
            self.lines[ident] = body.splitlines()
            self.kind[ident] = kind_of(body)
            self.nchars[ident] = len(body)
        self.idset = set(self.ids)
        self.parent: dict[str, str | None] = {}
        self.children: dict[str, list[str]] = defaultdict(list)
        for ident in self.ids:
            parent = ident.rsplit(".", 1)[0] if "." in ident else None
            if parent not in self.idset:
                parent = None
            self.parent[ident] = parent
            if parent is not None:
                self.children[parent].append(ident)
        self.short: dict[str, list[str]] = defaultdict(list)
        for ident in self.ids:
            self.short[ident.rsplit(".", 1)[-1]].append(ident)
        callers: dict[str, list[str]] = defaultdict(list)
        callees: dict[str, list[str]] = defaultdict(list)
        for edge in raw["edges"]:
            if edge.get("type") != "calls":
                continue
            src, dst = edge["source"], edge["target"]
            if src in self.idset and dst in self.idset and src != dst:
                callees[src].append(dst)
                callers[dst].append(src)
        self.callers = callers
        self.callees = callees
        self.first: dict[str, list[str]] = defaultdict(list)
        for ident, lines in self.lines.items():
            seen = set()
            for line in lines:
                if line in seen or not substantive(line):
                    continue
                seen.add(line)
                self.first[line].append(ident)
        archive = np.load(embed_path)
        rows = []
        ok = []
        self.emb_index: dict[str, int] = {}
        for ident in self.ids:
            if ident not in archive:
                continue
            vec = np.asarray(archive[ident], dtype=np.float64)
            if vec.shape != (256,):
                continue
            self.emb_index[ident] = len(rows)
            rows.append(vec)
            ok.append(ident)
        self.emb_ids = ok
        self.emb = np.vstack(rows) if rows else np.zeros((0, 256))
        self.mean = self.emb.mean(axis=0) if len(self.emb) else np.zeros(256)
        self.missing_emb = len(self.ids) - len(ok)
        archive.close()


def has_window(lines: list[str], seq: list[str]) -> bool:
    n = len(seq)
    if n == 0 or n > len(lines):
        return False
    first = seq[0]
    limit = len(lines) - n + 1
    start = 0
    while True:
        try:
            i = lines.index(first, start, limit)
        except ValueError:
            return False
        if lines[i : i + n] == seq:
            return True
        start = i + 1


def smallest_hit(graph: Graph, seq: list[str]) -> str | None:
    if not seq:
        return None
    # The line index stores substantive lines only. Windows often open on a
    # short token such as "):". Candidate nodes are those that contain the
    # first substantive line; the match is still the full consecutive window.
    key = next((line for line in seq if substantive(line)), None)
    if key is None:
        return None
    cands = graph.first.get(key, [])
    best = None
    best_len = None
    for ident in cands:
        if not has_window(graph.lines[ident], seq):
            continue
        nchars = graph.nchars[ident]
        if best is None or nchars < best_len or (nchars == best_len and ident < best):
            best = ident
            best_len = nchars
    return best


def line_support(graph: Graph, seq: list[str]) -> int:
    if len(seq) != 1:
        return 0
    return len(graph.first.get(seq[0], []))


def trim_ends(seq: list[str]) -> list[str]:
    start, end = 0, len(seq)
    while start < end and seq[start].strip() == "":
        start += 1
    while end > start and seq[end - 1].strip() == "":
        end -= 1
    return seq[start:end]


def usable_anchor(graph: Graph, seq: list[str]) -> bool:
    seq = trim_ends(seq)
    if not seq or not any(substantive(line) for line in seq):
        return False
    if len(seq) == 1 and line_support(graph, seq) > AMBIGUOUS_LINE_CAP:
        return False
    return True


def align_hunk(graph: Graph, hunk: dict) -> dict:
    deleted_w = [ln for ln in hunk["deleted"] if substantive(ln)]
    added_w = [ln for ln in hunk["added"] if substantive(ln)]
    path = hunk["path"]
    py = path.endswith(".py")
    weight = len(deleted_w) if deleted_w else len(added_w)
    if hunk["new_file"] or not py or weight == 0:
        return {
            "node": None,
            "weight": weight if py else len(deleted_w) + len(added_w),
            "how": "non_py" if not py else ("new_file" if hunk["new_file"] else "empty"),
            "py": py,
        }
    node = None
    how = "unaligned"
    window = trim_ends(hunk["old_lines"])
    if usable_anchor(graph, window):
        node = smallest_hit(graph, window)
        if node is not None:
            how = "old_window"
    if node is None and deleted_w:
        block = trim_ends(hunk["deleted"])
        if usable_anchor(graph, block):
            node = smallest_hit(graph, block)
            if node is not None:
                how = "deleted_block"
    return {"node": node, "weight": weight, "how": how if node else "unaligned", "py": True}


def issue_tokens(text: str) -> set[str]:
    tokens: set[str] = set()
    for tick in TICK_RE.findall(text):
        tokens.update(IDENT_RE.findall(tick))
        tokens.update(DOTTED_RE.findall(tick))
    tokens.update(DOTTED_RE.findall(text))
    tokens.update(CALL_RE.findall(text))
    tokens.update(SNAKE_RE.findall(text))
    tokens.update(CAMEL_RE.findall(text))
    kept = set()
    for token in tokens:
        if len(token) < 3:
            continue
        if token.lower() in KEYWORDS:
            continue
        kept.add(token)
    return kept


def resolve_seeds(graph: Graph, text: str) -> list[str]:
    found: list[str] = []
    seen = set()
    for token in sorted(issue_tokens(text)):
        hits: list[str] = []
        if token in graph.idset:
            hits = [token]
        else:
            short = token.rsplit(".", 1)[-1]
            pool = graph.short.get(short, [])
            if token.startswith(short) and token == short:
                if len(pool) > AMBIGUITY_CAP:
                    continue
                hits = list(pool)
            else:
                suffix = "." + token
                hits = [ident for ident in graph.ids if ident == token or ident.endswith(suffix)]
                if len(hits) > AMBIGUITY_CAP:
                    continue
        for ident in hits:
            if ident not in seen:
                seen.add(ident)
                found.append(ident)
    return found


def unit(vec: np.ndarray) -> np.ndarray | None:
    norm = float(np.linalg.norm(vec))
    if norm < 1e-12:
        return None
    return vec / norm


def residual_matrix(graph: Graph) -> tuple[np.ndarray, np.ndarray]:
    """Return residual row-vectors (unnormalized) aligned to graph.emb_ids, and a mask of usable rows."""
    rows = np.zeros_like(graph.emb)
    usable = np.ones(len(graph.emb_ids), dtype=bool)
    for i, ident in enumerate(graph.emb_ids):
        parent = graph.parent[ident]
        if parent is not None and parent in graph.emb_index:
            rows[i] = graph.emb[i] - graph.emb[graph.emb_index[parent]]
        else:
            rows[i] = graph.emb[i] - graph.mean
        if float(np.linalg.norm(rows[i])) < 1e-12:
            usable[i] = False
    return rows, usable


def topk_from_queries(
    graph: Graph,
    queries: list[np.ndarray],
    matrix: np.ndarray,
    usable: np.ndarray,
    exclude: set[str],
    k: int,
) -> set[str]:
    if k <= 0 or not queries or len(graph.emb_ids) == 0:
        return set()
    q = np.vstack(queries)
    qn = np.linalg.norm(q, axis=1, keepdims=True)
    mn = np.linalg.norm(matrix, axis=1, keepdims=True)
    good_q = (qn[:, 0] >= 1e-12)
    if not np.any(good_q):
        return set()
    q = q[good_q]
    qn = qn[good_q]
    sims = (q @ matrix.T) / (qn * mn.T + 1e-12)
    sims = np.nan_to_num(sims, nan=-np.inf, posinf=-np.inf, neginf=-np.inf)
    sims[:, ~usable] = -np.inf
    for ident in exclude:
        idx = graph.emb_index.get(ident)
        if idx is not None:
            sims[:, idx] = -np.inf
    # Best cosine any query assigns to a node, then top k.
    best = sims.max(axis=0)
    if k >= len(best):
        order = np.argsort(-best)
    else:
        part = np.argpartition(-best, k)[:k]
        order = part[np.argsort(-best[part])]
    chosen = []
    for idx in order:
        if not np.isfinite(best[idx]):
            continue
        chosen.append(graph.emb_ids[int(idx)])
        if len(chosen) >= k:
            break
    return set(chosen)


def arm_sets(graph: Graph, seeds: list[str], resid: np.ndarray, usable: np.ndarray) -> dict[str, set[str]]:
    seed_set = set(seeds)
    children: set[str] = set()
    siblings: set[str] = set()
    for ident in seeds:
        if graph.kind[ident] == "class":
            children.update(graph.children.get(ident, []))
        else:
            parent = graph.parent[ident]
            if parent is not None:
                siblings.update(graph.children.get(parent, []))
    siblings -= seed_set
    pset = children | siblings
    call: list[str] = []
    seen_call = set()
    for ident in seeds:
        for other in graph.callees.get(ident, []) + graph.callers.get(ident, []):
            if other in seed_set or other in seen_call:
                continue
            seen_call.add(other)
            call.append(other)
    k = len(pset) if pset else FIXED_K
    raw_queries = []
    res_queries = []
    for ident in seeds:
        idx = graph.emb_index.get(ident)
        if idx is None:
            continue
        raw_queries.append(graph.emb[idx])
        res_queries.append(resid[idx])
    ones = np.ones(len(graph.emb_ids), dtype=bool)
    return {
        "Z": set(seeds),
        "P": set(pset),
        "ZP": set(seeds) | set(pset),
        "C": set(call),
        "C50": set(call[:NEIGHBOR_CAP]),
        "E": topk_from_queries(graph, raw_queries, graph.emb, ones, seed_set, k),
        "R": topk_from_queries(graph, res_queries, resid, usable, seed_set, k),
        "_k": k,
        "_call_list": call,
    }


def cover(nodes: set[str], anchors: list[dict]) -> tuple[int, int, int]:
    """Return covered weight, alignable weight, task-hit among alignable hunks."""
    covered = 0
    alignable = 0
    hit = 0
    total = 0
    for anchor in anchors:
        alignable += anchor["weight"] if anchor["node"] else 0
        total += anchor["weight"]
        if anchor["node"] and anchor["node"] in nodes:
            covered += anchor["weight"]
            hit += 1
    return covered, alignable, hit


def random_cover(graph: Graph, exclude: set[str], k: int, anchors: list[dict], instance_id: str) -> float:
    pool = [ident for ident in graph.ids if ident not in exclude]
    if k <= 0 or not pool:
        return 0.0
    digest = hashlib.sha256(instance_id.encode()).hexdigest()
    rng = np.random.default_rng([RNG_SEED, int(digest[:8], 16)])
    draw_n = min(k, len(pool))
    alignable_nodes = [(a["node"], a["weight"]) for a in anchors if a["node"]]
    if not alignable_nodes:
        return 0.0
    acc = 0.0
    for _ in range(RANDOM_DRAWS):
        pick = set(rng.choice(pool, size=draw_n, replace=False).tolist())
        acc += sum(w for node, w in alignable_nodes if node in pick)
    return acc / RANDOM_DRAWS


def exclusive_bin(anchor_node: str, arms: dict[str, set[str]]) -> str:
    if anchor_node in arms["Z"]:
        return "named_body"
    if anchor_node in arms["P"]:
        return "sibling_or_child"
    if anchor_node in arms["C"]:
        return "call"
    if anchor_node in arms["R"]:
        return "residual_only"
    if anchor_node in arms["E"]:
        return "raw_cosine_only"
    return "none"


def micro(rows: list[dict], arm: str) -> dict:
    covered = sum(r["covered"][arm] for r in rows)
    alignable = sum(r["alignable"] for r in rows)
    all_w = sum(r["all_weight"] for r in rows)
    hits = sum(1 for r in rows if r["alignable"] and r["covered"][arm] > 0)
    tasks = sum(1 for r in rows if r["alignable"])
    return {
        "covered": covered,
        "alignable": alignable,
        "recall": (covered / alignable) if alignable else None,
        "recall_all": (covered / all_w) if all_w else None,
        "task_hits": hits,
        "tasks_with_anchor": tasks,
        "task_hit_rate": (hits / tasks) if tasks else None,
        "mean_set_size": float(np.mean([r["size"][arm] for r in rows])) if rows else None,
    }


def boot_diff(rows: list[dict], left: str, right: str, rng: np.random.Generator) -> dict:
    n = len(rows)
    if n == 0:
        return {"diff": None, "lo": None, "hi": None}
    point_l = micro(rows, left)["recall"]
    point_r = micro(rows, right)["recall"]
    point = None if point_l is None or point_r is None else point_l - point_r
    diffs = np.empty(N_BOOT)
    for i in range(N_BOOT):
        sample = [rows[j] for j in rng.integers(0, n, n)]
        a = micro(sample, left)["recall"]
        b = micro(sample, right)["recall"]
        diffs[i] = np.nan if a is None or b is None else a - b
    finite = diffs[np.isfinite(diffs)]
    if point is None or len(finite) == 0:
        return {"diff": point, "lo": None, "hi": None}
    lo, hi = np.quantile(finite, [0.025, 0.975])
    return {"diff": point, "lo": float(lo), "hi": float(hi)}


def main() -> None:
    OUT.mkdir(parents=True, exist_ok=True)
    cache: dict[str, Graph] = {}
    rows = []
    missing_graph = []
    how_counts: dict[str, int] = defaultdict(int)
    for line in TASKS.read_text().splitlines():
        if not line.strip():
            continue
        task = json.loads(line)
        instance = task["instance_id"]
        short = repo_short(task["repo"])
        commit = task["base_commit"]
        gpath = GRAPHS / f"{short}_{commit}.json"
        epath = EMBEDS / f"{short}_{commit}.npz"
        if not gpath.exists() or not epath.exists():
            missing_graph.append(instance)
            continue
        key = str(gpath)
        if key not in cache:
            cache[key] = Graph(gpath, epath)
            print(f"loaded {gpath.name} nodes={len(cache[key].ids)} emb_missing={cache[key].missing_emb}", file=sys.stderr)
        graph = cache[key]
        anchors = [align_hunk(graph, hunk) for hunk in parse_hunks(task["patch"])]
        for anchor in anchors:
            how_counts[anchor["how"]] += 1
        seeds = resolve_seeds(graph, task["problem_statement"])
        resid, usable = residual_matrix(graph)
        arms = arm_sets(graph, seeds, resid, usable)
        k = arms.pop("_k")
        arms.pop("_call_list")
        rand_covered = random_cover(graph, set(seeds), k, anchors, instance)
        covered = {}
        sizes = {}
        for name in ("Z", "P", "ZP", "C", "C50", "E", "R"):
            c, alignable, _hit = cover(arms[name], anchors)
            covered[name] = c
            sizes[name] = len(arms[name])
        covered["RAND"] = rand_covered
        sizes["RAND"] = k
        _, alignable, _ = cover(arms["Z"], anchors)
        all_weight = sum(a["weight"] for a in anchors)
        bins = []
        for anchor in anchors:
            if not anchor["node"] or anchor["weight"] == 0:
                continue
            bins.append({"node": anchor["node"], "weight": anchor["weight"], "bin": exclusive_bin(anchor["node"], arms), "how": anchor["how"]})
        held = "rich" if short == "rich" else "dev"
        rows.append(
            {
                "instance_id": instance,
                "repo": short,
                "split": held,
                "n_seeds": len(seeds),
                "seeds": seeds[:40],
                "k": k,
                "alignable": alignable,
                "all_weight": all_weight,
                "covered": covered,
                "size": sizes,
                "n_nodes": len(graph.ids),
                "n_with_parent": sum(1 for ident in graph.ids if graph.parent[ident]),
                "bins": bins,
                "zero_seed": len(seeds) == 0,
            }
        )
        print(f"{instance} seeds={len(seeds)} alignable={alignable} Z={covered['Z']} P={covered['P']} C={covered['C']} E={covered['E']} R={covered['R']}", file=sys.stderr)

    if missing_graph:
        raise SystemExit(f"missing graphs: {missing_graph[:8]}")

    def pool(name: str) -> list[dict]:
        if name == "dev":
            return [r for r in rows if r["split"] == "dev"]
        if name == "rich":
            return [r for r in rows if r["split"] == "rich"]
        if name == "all":
            return rows
        return [r for r in rows if r["repo"] == name]

    summary_pools = {}
    for name in ("dev", "rich", "fastapi", "requests", "httpx", "all"):
        group = pool(name)
        summary_pools[name] = {
            "tasks": len(group),
            "zero_seed_tasks": sum(1 for r in group if r["zero_seed"]),
            "alignable_lines": sum(r["alignable"] for r in group),
            "all_lines": sum(r["all_weight"] for r in group),
            "arms": {arm: micro(group, arm) for arm in PRIMARY_ARMS},
        }

    dev = pool("dev")
    ranking = sorted(
        ((summary_pools["dev"]["arms"][arm]["recall"] or -1, arm) for arm in WINNER_ARMS),
        reverse=True,
    )
    winner = ranking[0][1]
    rng = np.random.default_rng(RNG_SEED)
    contrasts = {}
    for split in ("dev", "rich"):
        group = pool(split)
        contrasts[split] = {
            "ZP_minus_Z": boot_diff(group, "ZP", "Z", rng),
            "R_minus_E": boot_diff(group, "R", "E", rng),
            "R_minus_C": boot_diff(group, "R", "C", rng),
            "Z_minus_C": boot_diff(group, "Z", "C", rng),
            "P_minus_C": boot_diff(group, "P", "C", rng),
        }

    bin_weight = defaultdict(lambda: defaultdict(int))
    for row in rows:
        for item in row["bins"]:
            bin_weight[row["split"]][item["bin"]] += item["weight"]
            # Rich is both a split and a repo. Count that weight once.
            if row["repo"] != row["split"]:
                bin_weight[row["repo"]][item["bin"]] += item["weight"]

    payload = {
        "protocol": {
            "ambiguity_cap": AMBIGUITY_CAP,
            "ambiguous_line_cap": AMBIGUOUS_LINE_CAP,
            "fixed_k": FIXED_K,
            "neighbor_cap_secondary": NEIGHBOR_CAP,
            "random_draws": RANDOM_DRAWS,
            "rng_seed": RNG_SEED,
            "min_line_chars": MIN_LINE_CHARS,
            "n_boot": N_BOOT,
            "winner_arms": list(WINNER_ARMS),
            "winner_rule": "max micro line-recall on non-Rich among winner arms",
            "issue_fields": ["problem_statement"],
            "ignored_fields": ["hints_text", "test_patch"],
        },
        "alignment_hunks": dict(how_counts),
        "winner_on_dev": winner,
        "dev_ranking": [{"arm": arm, "recall": rec} for rec, arm in ranking],
        "pools": summary_pools,
        "contrasts": contrasts,
        "exclusive_bins": {k: dict(v) for k, v in bin_weight.items()},
        "graph_cache": len(cache),
    }
    (OUT / "summary.json").write_text(json.dumps(payload, indent=2) + "\n")
    with (OUT / "per_task.jsonl").open("w") as handle:
        for row in rows:
            handle.write(json.dumps(row) + "\n")

    print(f"\nWinner on non-Rich: {winner}")
    header = f"{'pool':<10} {'lines':>7} " + " ".join(f"{arm:>8}" for arm in PRIMARY_ARMS)
    print(header)
    for name in ("dev", "fastapi", "requests", "httpx", "rich", "all"):
        block = summary_pools[name]
        cells = []
        for arm in PRIMARY_ARMS:
            rec = block["arms"][arm]["recall"]
            cells.append(f"{rec:8.3f}" if rec is not None else f"{'—':>8}")
        print(f"{name:<10} {block['alignable_lines']:7d} " + " ".join(cells))
    print("\nContrasts are micro-recall differences. Intervals are task bootstrap 95%.")
    for split, table in contrasts.items():
        print(split)
        for name, est in table.items():
            print(f"  {name}: {est['diff']:.3f} [{est['lo']:.3f}, {est['hi']:.3f}]" if est["diff"] is not None else f"  {name}: none")
    print("exclusive bins", {k: dict(v) for k, v in bin_weight.items()})
    print("alignment", dict(how_counts))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/repro/phase4/measure.py
"""Phase 4: SemDAG (in-sandbox rebuilt graph) vs the shipped graph, CPU only.

For each public task: extract the snapshot's .py files, build the SemDAG IR with
the UNMODIFIED semdag modules, and score repo_eval's localization ranking
against the gold edit functions of `patch`. `hints_text` and `test_patch` are
never read. `patch` is used for scoring only.

Gold edit site: the innermost function/method (pre-patch source, stdlib ast)
containing a deleted line; for pure-insertion hunks, the innermost function
containing BOTH old-side neighbours of the insertion. Anything else (new files,
module/class-level inserts, non-.py) is file-level gold only.

Arms (all ranked lists of SemDAG function/method nodes):
  A  repo_eval as shipped: PPR on the reversed call graph, seeded by issue
     symbols, taint seed blended at 0.25
  B  same, taint blend off
  S  seed nodes only, issue order
  SC seeds, then containment closure (methods of a class named in the issue;
     siblings of a seeded method)
Shipped graph (sets, not ranked): Z seeds, ZP seeds+outline, C one call hop.
"""
from __future__ import annotations

import ast
import json
import os
import random
import shutil
import sys
import tarfile
import tempfile
from collections import Counter, defaultdict
from multiprocessing import Pool
from pathlib import Path

HERE = Path(__file__).resolve().parent
ROOT = HERE.parents[1]
DATA = Path(os.environ.get("P4_DATA", ROOT / "DataSets/public/data"))
# SEMDAG: the graph-first system exactly as first shipped (frozen copy, audited in the paper).
# PROD: the repaired production code, the same files the v4_semdag skill ships.
SEMDAG = Path(os.environ.get("P4_SEMDAG_LEGACY", HERE / "legacy_semdag"))
PROD = Path(os.environ.get("P4_SEMDAG", ROOT / "AgenticSubmission/v4_semdag/skills/semdag/scripts"))
PHASE1 = Path(os.environ.get("P4_PHASE1", HERE.parent / "phase1"))
OUT = Path(os.environ.get("P4_OUT", HERE / "out"))
os.environ.setdefault("P4_SCRATCH", tempfile.gettempdir())
KS = (1, 5, 10)
DEPTH = 50
SKIP_PARTS = {".git", "node_modules", ".venv", "venv", "__pycache__", "site-packages"}


def is_test_path(rel: str) -> bool:
    parts = rel.split("/")
    name = parts[-1]
    return (any(p in ("tests", "test", "testing") for p in parts[:-1])
            or name.startswith("test_") or name.endswith("_test.py") or name == "conftest.py")


def parse_patch(patch: str):
    """Yield (path, is_new_file, hunks); hunk = list of (tag, old_lineno)."""
    files, cur, old = [], None, 0
    lines = patch.splitlines()
    i = 0
    while i < len(lines):
        ln = lines[i]
        if ln.startswith("--- "):
            a = ln[4:].strip()
            b = lines[i + 1][4:].strip() if i + 1 < len(lines) else ""
            path = b[2:] if b.startswith("b/") else b
            cur = {"path": path, "new": a == "/dev/null", "hunks": []}
            files.append(cur)
            i += 2
            continue
        if ln.startswith("@@") and cur is not None:
            try:
                old = int(ln.split()[1].split(",")[0][1:])
            except Exception:
                old = 1
            cur["hunks"].append([])
        elif cur is not None and cur["hunks"]:
            if ln.startswith("-"):
                cur["hunks"][-1].append(("-", old)); old += 1
            elif ln.startswith("+"):
                cur["hunks"][-1].append(("+", old))  # old = line AFTER the insert point
            elif ln.startswith(" ") or ln == "":
                cur["hunks"][-1].append((" ", old)); old += 1
        i += 1
    return files


def func_spans(src: str):
    """[(start, end, qualname, def_lineno, is_async)] for every def."""
    out = []
    try:
        tree = ast.parse(src)
    except SyntaxError:
        return out

    def walk(node, prefix):
        for ch in ast.iter_child_nodes(node):
            if isinstance(ch, ast.ClassDef):
                walk(ch, prefix + [ch.name])
            elif isinstance(ch, (ast.FunctionDef, ast.AsyncFunctionDef)):
                start = min([ch.lineno] + [d.lineno for d in ch.decorator_list])
                out.append((start, ch.end_lineno, ".".join(prefix + [ch.name]),
                            ch.lineno, isinstance(ch, ast.AsyncFunctionDef)))
                walk(ch, prefix + [ch.name])
            else:
                walk(ch, prefix)
    walk(tree, [])
    return out


def innermost(spans, line):
    best = None
    for s in spans:
        if s[0] <= line <= s[1] and (best is None or s[0] >= best[0]):
            best = s
    return best


def gold_sites(patch: str, repo_dir: str):
    gold, gold_files, n_hunks, n_aligned = {}, set(), 0, 0
    for f in parse_patch(patch):
        if not f["path"].endswith(".py"):
            continue
        fp = os.path.join(repo_dir, f["path"])
        if f["new"] or not os.path.isfile(fp):
            n_hunks += len(f["hunks"])
            continue
        gold_files.add(f["path"])
        spans = func_spans(open(fp, errors="replace").read())
        for h in f["hunks"]:
            n_hunks += 1
            dels = [o for t, o in h if t == "-"]
            hit = set()
            if dels:
                for o in dels:
                    s = innermost(spans, o)
                    if s:
                        hit.add(s)
            else:
                for o in {o for t, o in h if t == "+"}:
                    a, b = innermost(spans, o - 1), innermost(spans, o)
                    if a and b and a == b:
                        hit.add(a)
            if hit:
                n_aligned += 1
            for s in hit:
                gold[(f["path"], s[3])] = {"file": f["path"], "qual": s[2],
                                           "line": s[3], "async": s[4]}
    return gold, gold_files, n_hunks, n_aligned


def extract_py(tgz: Path, dest: str):
    with tarfile.open(tgz) as tf:
        members = []
        for m in tf:
            if not m.isfile() or not m.name.endswith(".py"):
                continue
            parts = Path(m.name).parts
            if any(p in SKIP_PARTS for p in parts):
                continue
            members.append(m)
        tf.extractall(dest, members=members, filter="data")


def shipped_arms(task, gold):
    """Set-based arms on the shipped graph, via the Phase 1 loader."""
    import importlib.util
    spec = importlib.util.spec_from_file_location("phase1_measure", PHASE1 / "measure.py")
    p1 = importlib.util.module_from_spec(spec); sys.modules["phase1_measure"] = p1; spec.loader.exec_module(p1)
    short = task["repo"].split("/")[-1]
    gp = DATA / "graphs" / f"{short}_{task['base_commit']}.json"
    ep = DATA / "embeddings" / f"{short}_{task['base_commit']}.npz"
    if not gp.exists() or gp.stat().st_size == 0:
        return None
    g = p1.Graph(gp, ep)

    def to_ids(site):
        dotted = site["file"][:-3].replace("/", ".") + "." + site["qual"]
        return {i for i in g.ids if i == dotted or dotted.endswith("." + i) and i.count(".") >= 1
                and i.endswith(site["qual"])}
    gold_ids = {k: to_ids(v) for k, v in gold.items()}
    seeds = p1.resolve_seeds(g, task["problem_statement"])
    z = set(seeds)
    p = set()
    for s in seeds:
        if g.kind[s] == "class":
            p.update(g.children.get(s, []))
        elif g.parent.get(s):
            p.update(g.children.get(g.parent[s], []))
    p -= z
    c = set()
    for s in seeds:
        c.update(g.callers.get(s, [])); c.update(g.callees.get(s, []))

    def hit(nodes):
        return any(ids & nodes for ids in gold_ids.values())
    return {"nodes": len(g.ids),
            "gold_present": sum(1 for ids in gold_ids.values() if ids),
            "gold_present_async": sum(1 for k, ids in gold_ids.items() if ids and gold[k]["async"]),
            "Z": {"hit": hit(z), "size": len(z)},
            "ZP": {"hit": hit(z | p), "size": len(z | p)},
            "C": {"hit": hit(c), "size": len(c)}}


def run_task(task):
    iid = task["instance_id"]
    tmp = tempfile.mkdtemp(prefix=f"p4_{iid}_", dir=os.environ["P4_SCRATCH"])
    os.environ["SEMDAG_CACHE"] = os.path.join(tmp, "_cache")
    repo = os.path.join(tmp, "repo")
    try:
        extract_py(DATA / "snapshots" / f"{iid}.tgz", repo)
        gold, gold_files, n_hunks, n_aligned = gold_sites(task["patch"], repo)

        sys.path.insert(0, str(SEMDAG))
        import cache, snapshot, semdag_ir_ast, taint, repo_eval
        from ppr_scorer import PurePythonPPRScorer
        cache.init_container(iid, repo)
        snap = snapshot.snapshot(iid, repo)
        ir = semdag_ir_ast.build(iid)
        par = repo_eval.parity_check(iid, snap, ir)
        tnt = taint.build(iid)
        sc = PurePythonPPRScorer(ir, tnt)
        syms = repo_eval.problem_symbols(task["problem_statement"])

        funcs = [n for n in sc.nodes if sc.kind.get(n) in ("function", "method")]
        key = {n: (sc.meta[n]["file"], sc.meta[n]["line"]) for n in funcs}
        node_keys = set(key.values())

        nontest = {n for n in funcs if not is_test_path(sc.meta[n]["file"])}
        diag = {}

        def ppr_rank(seed, tag=None):
            """Same ordering as PurePythonPPRScorer.score, without evidence columns.
            Returns (as shipped: all functions, same ranking restricted to non-test functions)."""
            tot = sum(seed.values()) or 1.0
            ppr = sc._ppr({k: v / tot for k, v in seed.items()})
            tp = sum(ppr.get(n, 0.0) for n in funcs) or 1.0
            rows = sorted(((-round(ppr.get(n, 0.0) / tp, 6), n) for n in funcs))
            if tag:
                allmass = sum(ppr.values()) or 1.0
                diag[tag] = {"mass_on_functions": round(tp / allmass, 4),
                             "mass_on_test_functions": round(sum(ppr.get(n, 0.0) for n in funcs if n not in nontest) / allmass, 4),
                             "mass_on_extern_or_ambiguous": round(sum(v for n, v in ppr.items() if sc.kind.get(n) in ("extern", "ambiguous")) / allmass, 4),
                             "test_fns_in_top10": sum(1 for _, n in rows[:10] if n not in nontest),
                             "test_fns_in_top5": sum(1 for _, n in rows[:5] if n not in nontest),
                             "tied_at_rank5": bool(len(rows) > 5 and rows[4][0] == rows[5][0])}
            return [n for _, n in rows[:DEPTH]], [n for _, n in rows if n in nontest][:DEPTH]

        seed_nodes = sc.names_to_nodes(syms)
        tseed = tnt.get("seed", {})
        sa = defaultdict(float)
        for n in seed_nodes:
            sa[n] += 1.0
        for n, w in tseed.items():
            sa[n] += 0.25 * w
        if not sa:
            sa = dict(tseed)
        sb = defaultdict(float)
        for n in seed_nodes:
            sb[n] += 1.0

        # containment from the IR's `contains` edges
        kids, parent = defaultdict(list), {}
        for e in ir["edges"]:
            if e["rel"] == "contains":
                kids[e["src"]].append(e["dst"]); parent[e["dst"]] = e["src"]
        cls_by_name = defaultdict(list)
        for n in ir["nodes"]:
            if n["kind"] == "class":
                cls_by_name[n["name"]].append(n["id"])
        s_list, seen = [], set()
        for n in seed_nodes:
            if n not in seen:
                seen.add(n); s_list.append(n)
        sc_list = list(s_list); seen2 = set(sc_list)
        for sym in syms:
            for cid in cls_by_name.get(sym, []):
                for k in kids.get(cid, []):
                    if k in key and k not in seen2:
                        seen2.add(k); sc_list.append(k)
        for n in s_list:
            p = parent.get(n)
            if p and sc.kind.get(p) == "class":
                for k in kids.get(p, []):
                    if k in key and k not in seen2:
                        seen2.add(k); sc_list.append(k)

        a_all, a_nt = ppr_rank(dict(sa), "A") if sa else ([], [])
        b_all, b_nt = ppr_rank(dict(sb), "B") if sb else ([], [])
        arms = {"A": a_all, "B": b_all, "S": s_list, "SC": sc_list,
                "Af": a_nt, "Bf": b_nt,
                "Sf": [n for n in s_list if n in nontest], "SCf": [n for n in sc_list if n in nontest]}
        n_edges = Counter(e["rel"] for e in ir["edges"])
        gkeys = set(gold)
        res = {}
        for name, lst in arms.items():
            keys = [key[n] for n in lst]
            files, sf = [], set()
            for f, _ in keys:
                if f not in sf:
                    sf.add(f); files.append(f)
            first = next((i + 1 for i, k in enumerate(keys) if k in gkeys), None)
            ffirst = next((i + 1 for i, f in enumerate(files) if f in gold_files), None)
            res[name] = {"n": len(lst), "first_fn": first, "first_file": ffirst}
        row = {"instance_id": iid, "repo": task["repo"], "parity_ok": par["ok"],
               "ir": ir["counts"], "n_funcs": len(funcs), "n_syms": len(syms),
               "n_seed_nodes": len(s_list), "n_seed_nodes_nontest": sum(1 for n in s_list if n in nontest),
               "n_funcs_nontest": len(nontest), "ppr_diag": diag, "ir_edges_by_rel": dict(n_edges),
               "hunks": n_hunks, "hunks_aligned": n_aligned,
               "gold": list(gold.values()), "gold_files": sorted(gold_files),
               "gold_in_semdag": sum(1 for k in gkeys if k in node_keys),
               "gold_async": sum(1 for v in gold.values() if v["async"]),
               "arms": res}
        try:
            row["shipped"] = shipped_arms(task, gold)
        except Exception as ex:  # report, never hide
            row["shipped"] = {"error": repr(ex)}
        return row
    except Exception as ex:
        return {"instance_id": iid, "repo": task["repo"], "error": repr(ex)}
    finally:
        shutil.rmtree(tmp, ignore_errors=True)


def summarize(rows):
    ok = [r for r in rows if "error" not in r]
    ev = [r for r in ok if r["gold"]]            # function-level evaluable
    evf = [r for r in ok if r["gold_files"]]     # file-level evaluable
    out = {"tasks": len(rows), "errors": [r["instance_id"] for r in rows if "error" in r],
           "parity_ok": sum(r["parity_ok"] for r in ok),
           "fn_evaluable": len(ev), "file_evaluable": len(evf),
           "hunks": sum(r["hunks"] for r in ok), "hunks_aligned": sum(r["hunks_aligned"] for r in ok),
           "gold_fns": sum(len(r["gold"]) for r in ev),
           "gold_in_semdag": sum(r["gold_in_semdag"] for r in ev),
           "gold_async": sum(r["gold_async"] for r in ev),
           "tasks_with_async_gold": sum(1 for r in ev if r["gold_async"]),
           "no_seed_tasks": sum(1 for r in ev if r["n_seed_nodes"] == 0)}
    sh = [r for r in ev if r.get("shipped") and "error" not in r["shipped"]]
    out["shipped"] = {"tasks": len(sh),
                      "gold_fns": sum(len(r["gold"]) for r in sh),
                      "gold_present": sum(r["shipped"]["gold_present"] for r in sh),
                      "gold_async": sum(r["gold_async"] for r in sh),
                      "gold_present_async": sum(r["shipped"]["gold_present_async"] for r in sh)}
    for a in ("Z", "ZP", "C"):
        out["shipped"][a] = {"hit": sum(r["shipped"][a]["hit"] for r in sh),
                             "mean_size": round(sum(r["shipped"][a]["size"] for r in sh) / max(1, len(sh)), 2)}
    out["arms"] = {}
    for a in ("A", "B", "S", "SC", "Af", "Bf", "Sf", "SCf"):
        d = {"mean_len": round(sum(r["arms"][a]["n"] for r in ev) / max(1, len(ev)), 2)}
        for k in KS:
            d[f"fn_hit@{k}"] = sum(1 for r in ev if (r["arms"][a]["first_fn"] or 99) <= k)
            d[f"file_hit@{k}"] = sum(1 for r in evf if (r["arms"][a]["first_file"] or 99) <= k)
        d["fn_hit@any"] = sum(1 for r in ev if r["arms"][a]["first_fn"])
        out["arms"][a] = d
    rng = random.Random(0)

    def boot(x, y, k=5):
        diffs = []
        for _ in range(4000):
            s = [ev[rng.randrange(len(ev))] for _ in ev]
            diffs.append(sum(((r["arms"][x]["first_fn"] or 99) <= k) - ((r["arms"][y]["first_fn"] or 99) <= k)
                             for r in s) / len(s))
        diffs.sort()
        pt = sum(((r["arms"][x]["first_fn"] or 99) <= k) - ((r["arms"][y]["first_fn"] or 99) <= k)
                 for r in ev) / len(ev)
        return {"diff": round(pt, 4), "ci95": [round(diffs[100], 4), round(diffs[3899], 4)]}
    if ev:
        out["contrasts_fn_hit@5"] = {"B-A": boot("B", "A"), "S-A": boot("S", "A"),
                                     "SC-S": boot("SC", "S"), "SC-B": boot("SC", "B")}
    by = defaultdict(list)
    for r in ev:
        by[r["repo"]].append(r)
    out["by_repo"] = {repo: {"n": len(rs), **{a: sum(1 for r in rs if (r["arms"][a]["first_fn"] or 99) <= 5)
                                              for a in ("A", "B", "S", "SC")}} for repo, rs in by.items()}
    return out


def main():
    OUT.mkdir(parents=True, exist_ok=True)
    tasks = [json.loads(l) for l in open(DATA / "tasks.jsonl")]
    for t in tasks:
        t.pop("hints_text", None); t.pop("test_patch", None)
    if len(sys.argv) > 1:
        tasks = [t for t in tasks if t["instance_id"] in sys.argv[1:]]
    with Pool(int(os.environ.get("P4_PROCS", "6"))) as pool:
        rows = pool.map(run_task, tasks, chunksize=1)
    with open(OUT / "per_task.jsonl", "w") as f:
        for r in rows:
            f.write(json.dumps(r, sort_keys=True) + "\n")
    s = summarize(rows)
    json.dump(s, open(OUT / "summary.json", "w"), indent=1, sort_keys=True)
    print(json.dumps(s, indent=1, sort_keys=True))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/repro/phase4/measure_v2.py
"""Phase 4c: an improved localizer on the rebuilt graph (SemDAG-L2).

Same gold definition and data handling as measure.py. `hints_text` and
`test_patch` are never read; `patch` is scoring only.

Fixed design choices (set before any score was read; none tuned on gold):
  - candidates: functions/methods outside test files (tests/ dirs, test_*.py,
    *_test.py, conftest.py)
  - N   name seeds: `Class.method` dotted mentions, bare function names, then
        methods of classes named in the issue, then functions of files named
        in the issue (path or module mention)
  - X   BM25 (k1=1.2, b=0.75) over function documents: identifier tokens of the
        body, with qualname and file-path tokens repeated 3x
  - G   Personalized PageRank (d=0.85, 40 iters) on the UNDIRECTED calls+contains
        graph, seeded by name seeds (1.0 each) plus the BM25 top-20 (normalised)
  - fusion: reciprocal rank fusion, k=60
Arms: N, X, NX = RRF(N,X), NXG = RRF(N,X,G).
Selection: the arm with the highest function Hit@5 on FastAPI+Requests+httpx.
Rich is held out and printed, not used to choose.
"""
from __future__ import annotations

import json
import math
import os
import random
import re
import shutil
import sys
import tempfile
from collections import Counter, defaultdict
from multiprocessing import Pool
from pathlib import Path

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))
import measure as m1  # noqa: E402

OUT = m1.OUT
ARMS = ("N", "X", "NX", "NXG", "XF", "EX", "D15", "D30", "D50", "D85", "DC85")
KS = (1, 5, 10)
DEPTH = 50
IDENT = re.compile(r"[A-Za-z_][A-Za-z0-9_]*")
DOTTED = re.compile(r"[A-Za-z_][A-Za-z0-9_]*(?:\.[A-Za-z_][A-Za-z0-9_]*)+")
PATHISH = re.compile(r"[A-Za-z0-9_\-./]+\.py")
CAMEL = re.compile(r"[A-Z]+(?=[A-Z][a-z])|[A-Z]?[a-z]+|[A-Z]+|\d+")
STOP = {"the", "and", "for", "that", "this", "with", "not", "but", "are", "was", "from", "have",
        "has", "when", "you", "can", "should", "would", "will", "use", "using", "def", "self",
        "return", "import", "none", "true", "false", "class", "if", "else", "in", "is", "of",
        "to", "it", "as", "on", "be", "or", "an", "at", "by", "we"}


def is_test_path(rel: str) -> bool:
    parts = rel.split("/")
    name = parts[-1]
    return (any(p in ("tests", "test", "testing") for p in parts[:-1])
            or name.startswith("test_") or name.endswith("_test.py") or name == "conftest.py")


def toks(text: str):
    out = []
    for ident in IDENT.findall(text):
        low = ident.lower()
        if len(low) >= 2 and low not in STOP:
            out.append(low)
        parts = [p.lower() for chunk in ident.split("_") for p in CAMEL.findall(chunk)]
        if len(parts) > 1:
            out.extend(p for p in parts if len(p) >= 2 and p not in STOP)
    return out


def rrf(lists, k=60):
    score = defaultdict(float)
    for lst in lists:
        for i, x in enumerate(lst):
            score[x] += 1.0 / (k + i + 1)
    return [x for x, _ in sorted(score.items(), key=lambda kv: (-kv[1], kv[0]))]


def run_task(task):
    iid = task["instance_id"]
    tmp = tempfile.mkdtemp(prefix=f"p4b_{iid}_", dir=os.environ["P4_SCRATCH"])
    repo = os.path.join(tmp, "repo")
    try:
        m1.extract_py(m1.DATA / "snapshots" / f"{iid}.tgz", repo)
        gold, gold_files, _, _ = m1.gold_sites(task["patch"], repo)
        issue = task["problem_statement"]

        # ---- function table straight from source (stdlib ast) ----
        funcs = []  # dict(key, file, qual, name, cls, start, end, doc_tokens)
        by_file = defaultdict(list)
        for root, _, names in os.walk(repo):
            for nm in sorted(names):
                if not nm.endswith(".py"):
                    continue
                fp = os.path.join(root, nm)
                rel = os.path.relpath(fp, repo)
                if is_test_path(rel) and not os.environ.get("P4_KEEP_TESTS"):
                    continue
                src = open(fp, errors="replace").read()
                lines = src.splitlines()
                for start, end, qual, defline, _async in m1.func_spans(src):
                    body = "\n".join(lines[start - 1:end])
                    head = " ".join([qual.replace(".", " "), rel.replace("/", " ").replace(".py", "")])
                    doc = toks(body) + toks(head) * 3
                    f = {"key": (rel, defline), "file": rel, "qual": qual,
                         "name": qual.rsplit(".", 1)[-1],
                         "cls": qual.rsplit(".", 1)[0] if "." in qual else None,
                         "start": start, "end": end, "doc": Counter(doc), "len": len(doc)}
                    by_file[rel].append(len(funcs)); funcs.append(f)
        n = len(funcs)
        if n == 0:
            return {"instance_id": iid, "repo": task["repo"], "error": "no functions"}
        by_name, by_qual_suffix, by_cls = defaultdict(list), defaultdict(list), defaultdict(list)
        for i, f in enumerate(funcs):
            by_name[f["name"]].append(i)
            parts = f["qual"].split(".")
            if len(parts) >= 2:
                by_qual_suffix[".".join(parts[-2:])].append(i)
                by_cls[parts[-2]].append(i)

        # ---- N: name seeds ----
        N, seen = [], set()

        def push(idxs):
            for i in idxs:
                if i not in seen:
                    seen.add(i); N.append(i)
        dotted = DOTTED.findall(issue)
        idents = list(dict.fromkeys(IDENT.findall(issue)))
        for d in dotted:
            parts = d.split(".")
            if len(parts) >= 2:
                push(by_qual_suffix.get(".".join(parts[-2:]), []))
        for w in idents:
            if len(w) >= 3 and len(by_name.get(w, [])) <= 20:
                push(by_name.get(w, []))
        for w in idents:
            if len(w) >= 3 and w[:1].isupper() and len(by_cls.get(w, [])) <= 60:
                push(by_cls.get(w, []))
        mentioned_files = set()
        for p in PATHISH.findall(issue):
            p = p.lstrip("./")
            for rel in by_file:
                if rel == p or rel.endswith("/" + p):
                    mentioned_files.add(rel)
        for d in dotted:
            p = d.replace(".", "/")
            for rel in by_file:
                stem = rel[:-3]
                if stem == p or stem.endswith("/" + p) or p.endswith("/" + stem.split("/")[-1]) and stem.endswith(p.split("/")[-2] + "/" + stem.split("/")[-1]) if "/" in p else False:
                    mentioned_files.add(rel)
        for rel in sorted(mentioned_files):
            push(by_file[rel])

        # ---- X: BM25 ----
        df = Counter()
        for f in funcs:
            df.update(f["doc"].keys())
        avg = sum(f["len"] for f in funcs) / n
        q = Counter(toks(issue))
        scores = [0.0] * n
        for term, qtf in q.items():
            d = df.get(term)
            if not d:
                continue
            idf = math.log(1 + (n - d + 0.5) / (d + 0.5))
            w = idf * min(qtf, 3)
            for i, f in enumerate(funcs):
                tf = f["doc"].get(term)
                if tf:
                    scores[i] += w * tf * 2.2 / (tf + 1.2 * (0.25 + 0.75 * f["len"] / avg))
        order = sorted(range(n), key=lambda i: (-scores[i], funcs[i]["key"]))
        X = [i for i in order if scores[i] > 0][:200]

        # ---- G: PPR on undirected calls+contains ----
        adj = defaultdict(set)
        for idxs in by_cls.values():           # contains: methods of one class (same file)
            groups = defaultdict(list)
            for i in idxs:
                groups[(funcs[i]["file"], funcs[i]["cls"])].append(i)
            for g in groups.values():
                if len(g) <= 80:
                    for a in g:
                        adj[a].update(x for x in g if x != a)
        call_re = re.compile(r"([A-Za-z_][A-Za-z0-9_]*)\s*\(")
        for i, f in enumerate(funcs):          # calls: uniquely resolvable bare names only
            for nm in set(f["doc"].keys()):
                pass
        # cheap call edges from the system's own IR resolution policy: unique global name
        name_unique = {nm: idxs[0] for nm, idxs in by_name.items() if len(idxs) == 1}
        src_cache = {}
        for i, f in enumerate(funcs):
            if f["file"] not in src_cache:
                src_cache[f["file"]] = open(os.path.join(repo, f["file"]), errors="replace").read().splitlines()
            body = "\n".join(src_cache[f["file"]][f["start"] - 1:f["end"]])
            for nm in set(call_re.findall(body)):
                j = name_unique.get(nm)
                if j is not None and j != i:
                    adj[i].add(j); adj[j].add(i)
        seed = defaultdict(float)
        for i in N[:50]:
            seed[i] += 1.0
        top = X[:20]
        if top:
            mx = scores[top[0]] or 1.0
            for i in top:
                seed[i] += scores[i] / mx
        G = []
        if seed:
            tot = sum(seed.values())
            p = {i: v / tot for i, v in seed.items()}
            r = dict(p)
            for _ in range(40):
                nxt = defaultdict(float)
                for i, v in r.items():
                    nb = adj.get(i)
                    if nb:
                        share = 0.85 * v / len(nb)
                        for j in nb:
                            nxt[j] += share
                    else:
                        for j, pv in p.items():
                            nxt[j] += 0.85 * v * pv
                for j, pv in p.items():
                    nxt[j] += 0.15 * pv
                r = nxt
            G = [i for i, _ in sorted(r.items(), key=lambda kv: (-kv[1], funcs[kv[0]]["key"]))][:200]

        # ---- XF: fuse function BM25 with a file-level BM25 ordering ----
        fdoc = {rel: Counter() for rel in by_file}
        for f in funcs:
            fdoc[f["file"]].update(f["doc"])
        fn_files = len(fdoc)
        fdf = Counter()
        for c in fdoc.values():
            fdf.update(c.keys())
        flen = {rel: sum(c.values()) for rel, c in fdoc.items()}
        favg = sum(flen.values()) / max(1, fn_files)
        fscore = defaultdict(float)
        for term, qtf in q.items():
            d = fdf.get(term)
            if not d:
                continue
            idf = math.log(1 + (fn_files - d + 0.5) / (d + 0.5))
            for rel, c in fdoc.items():
                tf = c.get(term)
                if tf:
                    fscore[rel] += idf * min(qtf, 3) * tf * 2.2 / (tf + 1.2 * (0.25 + 0.75 * flen[rel] / favg))
        file_order = sorted(fdoc, key=lambda rel: (-fscore[rel], rel))
        frank = {rel: i for i, rel in enumerate(file_order)}
        FO = sorted(X, key=lambda i: (frank[funcs[i]["file"]], -scores[i], funcs[i]["key"]))
        XF = rrf([X, FO])

        # ---- EX: explicit code-style mentions first, then BM25 ----
        codeish = set(re.findall(r"`([^`\n]+)`", issue))
        ex_names = set()
        for c in codeish:
            ex_names.update(IDENT.findall(c))
        ex_names.update(re.findall(r"([A-Za-z_][A-Za-z0-9_]{2,})\s*\(", issue))
        ex_names.update(re.findall(r"\.([A-Za-z_][A-Za-z0-9_]{2,})", issue))
        E = [i for i in X if funcs[i]["name"] in ex_names and len(by_name[funcs[i]["name"]]) <= 3]
        eset = set(E)
        EX = E + [i for i in X if i not in eset]

        # ---- D: production retriever, shared byte-for-byte with repo_eval ----
        os.environ["SEMDAG_CACHE"] = os.path.join(tmp, "_cache")
        sys.path.insert(0, str(m1.PROD))
        import cache as _cache, snapshot as _snapshot, semdag_ir_ast as _irmod, retriever as _retriever
        _cache.init_container(iid, repo)
        snap = _snapshot.snapshot(iid, repo)
        ir = _irmod.build(iid)
        idx_of = {f["key"]: i for i, f in enumerate(funcs)}
        node_to_idx = {nd["id"]: idx_of[(nd["file"], nd["line"])] for nd in ir["nodes"]
                       if nd["kind"] in ("function", "method") and (nd["file"], nd["line"]) in idx_of}

        def production_order(damping, weights=None, direction="undirected"):
            result = _retriever.rank(ir, snap, issue, 200, damping=damping,
                                     include_tests=bool(os.environ.get("P4_KEEP_TESTS")),
                                     relation_weights=weights, direction=direction)
            return [node_to_idx[s["id"]] for s in result["top"] if s["id"] in node_to_idx], result

        shared_x, x_receipt = production_order(0.0)
        # With tests kept, the production snapshot omits files over 256 KB (large test modules),
        # so the candidate sets differ and the identity is only asserted in the default mode.
        if not os.environ.get("P4_KEEP_TESTS") and \
                [funcs[i]["key"] for i in shared_x] != [funcs[i]["key"] for i in X]:
            raise AssertionError("measurement BM25 diverged from production retriever")
        D, d_receipts = {}, {}
        for damping in (0.15, 0.30, 0.50, 0.85):
            name = f"D{int(damping * 100)}"
            D[name], d_receipts[name] = production_order(damping)
        D["DC85"], d_receipts["DC85"] = production_order(
            0.85, {"contains": 0.0, "calls": 1.0, "imports": 0.0}, "reverse_calls")

        lists = {"N": N, "X": X, "NX": rrf([N, X]), "NXG": rrf([N, X, G]), "XF": XF, "EX": EX, **D}

        # ---- control: the same BM25 over the SHIPPED graph's node texts ----
        shipped = None
        try:
            def is_func(text):
                for line in text.splitlines():
                    st = line.lstrip()
                    if not st or st.startswith("@"):
                        continue
                    return not st.startswith("class ")
                return True
            short = task["repo"].split("/")[-1]
            gp = m1.DATA / "graphs" / f"{short}_{task['base_commit']}.json"
            raw = json.loads(gp.read_text())
            snodes = [(nd["id"], nd.get("text") or "") for nd in raw["nodes"]]
            snodes = [(i, t) for i, t in snodes
                      if not (i.startswith(("tests.", "test.")) or ".tests." in i or ".test_" in i
                              or i.split(".")[-1].startswith("test_") or "conftest" in i)
                      and is_func(t)]
            sdocs = [Counter(toks(t) + toks(i.replace(".", " ")) * 3) for i, t in snodes]
            sn = len(sdocs)
            sdf = Counter()
            for c in sdocs:
                sdf.update(c.keys())
            slen = [sum(c.values()) for c in sdocs]
            savg = sum(slen) / max(1, sn)
            ssc = [0.0] * sn
            for term, qtf in q.items():
                d = sdf.get(term)
                if not d:
                    continue
                idf = math.log(1 + (sn - d + 0.5) / (d + 0.5))
                for j, c in enumerate(sdocs):
                    tf = c.get(term)
                    if tf:
                        ssc[j] += idf * min(qtf, 3) * tf * 2.2 / (tf + 1.2 * (0.25 + 0.75 * slen[j] / savg))
            sorder = [j for j in sorted(range(sn), key=lambda j: (-ssc[j], snodes[j][0])) if ssc[j] > 0][:DEPTH]
            ids = [snodes[j][0] for j in sorder]
            gold_ids = set()
            for site in gold.values():
                dotted = site["file"][:-3].replace("/", ".") + "." + site["qual"]
                gold_ids.update(i for i, _ in snodes if i == dotted or (dotted.endswith("." + i) and i.endswith(site["qual"]) and "." in i))
            shipped = {"candidates": sn, "gold_ids": len(gold_ids),
                       "first_fn": next((r + 1 for r, i in enumerate(ids) if i in gold_ids), None)}
        except Exception as ex:
            shipped = {"error": repr(ex)}
        gkeys = set(gold)
        res = {}
        for name, lst in lists.items():
            lst = lst[:DEPTH]
            keys = [funcs[i]["key"] for i in lst]
            files, sf = [], set()
            for fkey, _ in keys:
                if fkey not in sf:
                    sf.add(fkey); files.append(fkey)
            res[name] = {"n": len(lst),
                         "first_fn": next((i + 1 for i, k in enumerate(keys) if k in gkeys), None),
                         "first_file": next((i + 1 for i, f in enumerate(files) if f in gold_files), None),
                         "top5": [f"{funcs[i]['file']}::{funcs[i]['qual']}" for i in lst[:5]]}
        cand_keys = {f["key"] for f in funcs}
        return {"instance_id": iid, "repo": task["repo"], "n_candidates": n,
                "gold_n": len(gkeys), "gold_in_candidates": sum(1 for k in gkeys if k in cand_keys),
                "gold_files": len(gold_files),
                "gold_test_only": bool(gkeys) and not any(k in cand_keys for k in gkeys),
                "async_gold": any(v["async"] for v in gold.values()),
                "dag": {"nodes": ir["counts"]["nodes"], "fn_nodes": len(node_to_idx),
                        "edges": {k: ir["counts"].get(k, 0) for k in ("contains", "calls", "imports")},
                        "production_receipts": {k: {"damping": v["damping"],
                                                     "relations_used": v["relations_used"]}
                                                for k, v in d_receipts.items()}},
                "shipped_bm25": shipped, "arms": res}
    except Exception as ex:
        import traceback
        return {"instance_id": iid, "repo": task["repo"], "error": repr(ex), "tb": traceback.format_exc()[-600:]}
    finally:
        shutil.rmtree(tmp, ignore_errors=True)


def hit(r, arm, k, level="fn"):
    return ((r["arms"][arm][f"first_{level}"] or 10 ** 6) <= k)


def table(rows, arms):
    ev = [r for r in rows if r["gold_n"]]
    evf = [r for r in rows if r["gold_files"]]
    out = {"fn_tasks": len(ev), "file_tasks": len(evf)}
    for a in arms:
        d = {}
        for k in KS:
            d[f"fn@{k}"] = sum(hit(r, a, k) for r in ev)
            d[f"file@{k}"] = sum(hit(r, a, k, "file") for r in evf)
        d["mrr_fn"] = round(sum(1.0 / r["arms"][a]["first_fn"] for r in ev if r["arms"][a]["first_fn"]) / max(1, len(ev)), 4)
        out[a] = d
    return out


def main():
    OUT.mkdir(parents=True, exist_ok=True)
    tasks = [json.loads(l) for l in open(m1.DATA / "tasks.jsonl")]
    for t in tasks:
        t.pop("hints_text", None); t.pop("test_patch", None)
    if len(sys.argv) > 1:
        tasks = [t for t in tasks if t["instance_id"] in sys.argv[1:]]
    with Pool(int(os.environ.get("P4_PROCS", "6"))) as pool:
        rows = pool.map(run_task, tasks, chunksize=1)
    with open(OUT / "v2_per_task.jsonl", "w") as f:
        for r in rows:
            f.write(json.dumps(r, sort_keys=True) + "\n")
    errs = [r for r in rows if "error" in r]
    ok = [r for r in rows if "error" not in r]
    # join the shipped-system arm A from measure.py for paired contrasts
    base = {}
    p1 = OUT / "per_task.jsonl"
    if p1.exists():
        for l in open(p1):
            r = json.loads(l)
            if "arms" in r:
                base[r["instance_id"]] = r["arms"]["A"]
    for r in ok:
        if r["instance_id"] in base:
            r["arms"]["A"] = base[r["instance_id"]]
    arms = ARMS + (("A",) if all("A" in r["arms"] for r in ok) else ())
    sel = [r for r in ok if not r["repo"].endswith("rich")]
    held = [r for r in ok if r["repo"].endswith("rich")]
    s = {"tasks": len(rows), "errors": [(r["instance_id"], r["error"]) for r in errs],
         "gold_fns": sum(r["gold_n"] for r in ok),
         "gold_in_candidates": sum(r["gold_in_candidates"] for r in ok),
         "tasks_gold_only_in_tests": sum(1 for r in ok if r["gold_test_only"]),
         "median_candidates": sorted(r["n_candidates"] for r in ok)[len(ok) // 2] if ok else 0,
         "selection": table(sel, arms), "heldout_rich": table(held, arms), "all": table(ok, arms)}
    def ship(pool):
        ev = [r for r in pool if r["gold_n"]]
        ok_ = [r for r in ev if r["shipped_bm25"] and "error" not in r["shipped_bm25"]]
        return {"tasks": len(ev), "scored": len(ok_),
                **{f"fn@{k}": sum(1 for r in ok_ if (r["shipped_bm25"]["first_fn"] or 10 ** 6) <= k) for k in KS}}
    s["shipped_bm25"] = {"selection": ship(sel), "heldout_rich": ship(held), "all": ship(ok)}
    asy = [r for r in ok if r.get("async_gold")]
    s["async_gold_tasks"] = {"n": len(asy), **{a: [sum(hit(r, a, k) for r in asy) for k in KS] for a in arms},
                             "shipped_bm25": [sum(1 for r in asy if r["shipped_bm25"] and (r["shipped_bm25"].get("first_fn") or 10 ** 6) <= k) for k in KS]}
    sel_ev = [r for r in sel if r["gold_n"]]
    s["winner"] = max(ARMS, key=lambda a: (sum(hit(r, a, 5) for r in sel_ev), -ARMS.index(a)))
    rng = random.Random(0)

    def boot(pool, x, y, k):
        ev = [r for r in pool if r["gold_n"]]
        if not ev or y not in ev[0]["arms"]:
            return None
        pt = sum(hit(r, x, k) - hit(r, y, k) for r in ev) / len(ev)
        ds = []
        for _ in range(4000):
            smp = [ev[rng.randrange(len(ev))] for _ in ev]
            ds.append(sum(hit(r, x, k) - hit(r, y, k) for r in smp) / len(smp))
        ds.sort()
        return {"diff": round(pt, 4), "ci95": [round(ds[100], 4), round(ds[3899], 4)]}
    w = s["winner"]
    best_d = max(("D15", "D30", "D50", "D85"), key=lambda a: (sum(hit(r, a, 5) for r in sel_ev), -int(a[1:])))
    s["best_dag_arm_on_selection"] = best_d
    s["dag_contrasts"] = {f"{best_d}-X fn@5 selection": boot(sel, best_d, "X", 5), f"{best_d}-X fn@5 heldout": boot(held, best_d, "X", 5),
                          f"{best_d}-X fn@10 heldout": boot(held, best_d, "X", 10), f"{best_d}-X fn@5 all": boot(ok, best_d, "X", 5),
                          f"{best_d}-X fn@1 all": boot(ok, best_d, "X", 1), f"{best_d}-X fn@10 all": boot(ok, best_d, "X", 10),
                          "DC85-X fn@5 all": boot(ok, "DC85", "X", 5)}
    s["contrasts"] = {f"{w}-A fn@5 all": boot(ok, w, "A", 5), f"{w}-A fn@10 all": boot(ok, w, "A", 10),
                      f"{w}-A fn@5 heldout": boot(held, w, "A", 5), f"{w}-A fn@10 heldout": boot(held, w, "A", 10),
                      f"{w}-X fn@5 all": boot(ok, w, "X", 5), f"{w}-N fn@5 all": boot(ok, w, "N", 5)}
    json.dump(s, open(OUT / "v2_summary.json", "w"), indent=1, sort_keys=True)
    print(json.dumps(s, indent=1, sort_keys=True))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/repro/phase4/equal_footing.py
"""Paired bootstrap contrasts on equal footing: both arms with the test-file filter (X vs Af),
and both without it (X with tests kept vs A as shipped). Reads the outputs of measure.py,
measure_v2.py, and a P4_KEEP_TESTS=1 run of measure_v2.py written to <out>_keep_tests.
"""
import json
import os
import random
import statistics
from pathlib import Path

HERE = Path(__file__).resolve().parent
OUT = Path(os.environ.get("P4_OUT", HERE / "out"))
KEEP = Path(os.environ.get("P4_OUT_KEEP", str(OUT) + "_keep_tests"))


def load(p):
    return {json.loads(l)["instance_id"]: json.loads(l) for l in open(p)}


v1 = {i: r for i, r in load(OUT / "per_task.jsonl").items() if "arms" in r}
v2 = load(OUT / "v2_per_task.jsonl")
kt = load(KEEP / "v2_per_task.jsonl")
ev = [i for i, r in v2.items() if r["gold_n"]]
held = [i for i in ev if v2[i]["repo"].endswith("rich")]
hit = lambda v, k: (v or 10 ** 6) <= k
get = {"X": lambda i: v2[i]["arms"]["X"]["first_fn"], "D15": lambda i: v2[i]["arms"]["D15"]["first_fn"],
       "Af": lambda i: v1[i]["arms"]["Af"]["first_fn"], "A": lambda i: v1[i]["arms"]["A"]["first_fn"],
       "Xall": lambda i: kt[i]["arms"]["X"]["first_fn"]}
rng = random.Random(0)


def boot(ids, x, y, k):
    pt = sum(hit(get[x](i), k) - hit(get[y](i), k) for i in ids) / len(ids)
    ds = []
    for _ in range(4000):
        smp = [ids[rng.randrange(len(ids))] for _ in ids]
        ds.append(sum(hit(get[x](i), k) - hit(get[y](i), k) for i in smp) / len(smp))
    ds.sort()
    return {"diff": round(pt, 4), "ci95": [round(ds[100], 4), round(ds[3899], 4)]}


count = lambda arm, ids: [sum(hit(get[arm](i), k) for i in ids) for k in (1, 5, 10)]
dg = [r["ppr_diag"]["A"] for r in v1.values() if r.get("gold") and "A" in r["ppr_diag"]]
out = {"X-Af fn@5 all (both filtered)": boot(ev, "X", "Af", 5), "X-Af fn@5 heldout": boot(held, "X", "Af", 5),
       "X-Af fn@10 all": boot(ev, "X", "Af", 10),
       "Xall-A fn@5 all (both unfiltered)": boot(ev, "Xall", "A", 5), "Xall-A fn@5 heldout": boot(held, "Xall", "A", 5),
       "D15-Af fn@5 all": boot(ev, "D15", "Af", 5),
       "counts": {"Af_all": count("Af", ev), "Af_heldout": count("Af", held),
                  "Af_selection": count("Af", [i for i in ev if i not in held]),
                  "Xall_all": count("Xall", ev), "Xall_heldout": count("Xall", held), "A_heldout": count("A", held)},
       "ppr_A": {"mean_mass_on_test_functions": round(statistics.mean(d["mass_on_test_functions"] for d in dg), 3),
                 "mean_test_fns_in_top10": round(statistics.mean(d["test_fns_in_top10"] for d in dg), 2)}}
json.dump(out, open(OUT / "equal_footing.json", "w"), indent=1)
print(json.dumps(out, indent=1))


In [ ]:
%%writefile /kaggle/working/repro/phase4/verify_gate.py
"""Phase 4b: the repo_eval --verify gate, measured without a model call.

For each public task: extract the full snapshot, commit it, run repo_eval once
(the frozen baseline, exactly the coder's first command), then for each
candidate patch apply it, run `repo_eval.py t . --verify` (no --test: syntax
and change detection only), record the verdict, and reset the tree.

Candidates: the gold `patch` (all 129 tasks; a correct gate must say clean),
and the Phase 2 E4B one-turn diffs (Rich, arms O and C, copies skipped).
`test_patch` and `hints_text` are never read.
"""
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import tempfile
from multiprocessing import Pool
from pathlib import Path

HERE = Path(__file__).resolve().parent
ROOT = HERE.parents[1]
DATA = Path(os.environ.get("P4_DATA", ROOT / "DataSets/public/data"))
# Default: the repaired production gate. Set P4_SEMDAG=<phase4_semdag/legacy_semdag> for the as-shipped gate.
REPO_EVAL = Path(os.environ.get("P4_SEMDAG", ROOT / "AgenticSubmission/v4_semdag/skills/semdag/scripts")) / "repo_eval.py"
COMPL = Path(os.environ.get("P4_COMPLETIONS", HERE.parent / "phase2/out/completions"))
OUT = Path(os.environ.get("P4_OUT", HERE / "out"))
os.environ.setdefault("P4_SCRATCH", tempfile.gettempdir())


def sh(cmd, cwd, env=None, inp=None, timeout=300):
    p = subprocess.run(cmd, cwd=cwd, env=env, input=inp, capture_output=True, text=True, timeout=timeout)
    return p.returncode, p.stdout, p.stderr


def run_task(task):
    iid = task["instance_id"]
    tmp = tempfile.mkdtemp(prefix=f"p4v_{iid}_", dir=os.environ["P4_SCRATCH"])
    repo = os.path.join(tmp, "repo"); os.makedirs(repo)
    env = dict(os.environ, SEMDAG_CACHE=os.path.join(tmp, "_cache"), PYTHONDONTWRITEBYTECODE="1")
    git = ["git", "-c", "user.name=p4", "-c", "user.email=p4@example.invalid"]
    row = {"instance_id": iid, "repo": task["repo"], "candidates": []}
    try:
        sh(["tar", "-xzf", str(DATA / "snapshots" / f"{iid}.tgz"), "-C", repo], tmp)
        if not os.path.isdir(os.path.join(repo, ".git")):
            sh(git + ["init", "-q"], repo)
        sh(git + ["add", "-A"], repo); sh(git + ["commit", "-qm", "p4base", "--allow-empty"], repo)
        rc, out, err = sh([sys.executable, str(REPO_EVAL), "t", ".",
                           "--problem", task["problem_statement"]], repo, env)
        try:
            base = json.loads(out)
            row["baseline"] = {"parity_ok": base["parity_ok"], "top_edit_sites": len(base["top_edit_sites"])}
        except Exception:
            row["baseline"] = {"error": (err or out)[-300:]}

        cands = [("gold", task["patch"])]
        for arm in ("O", "C"):
            fp = COMPL / f"{iid}_{arm}.json"
            if fp.exists():
                d = json.loads(fp.read_text())
                if d.get("copied_from_identical_prompt") or d.get("no_diff") or not d.get("diff"):
                    row["candidates"].append({"name": f"e4b_{arm}", "skipped":
                        "copy" if d.get("copied_from_identical_prompt") else "no_diff"})
                    continue
                cands.append((f"e4b_{arm}", d["diff"]))
        for name, diff in cands:
            if not diff.endswith("\n"):
                diff += "\n"
            c = {"name": name}
            rc, _, err = sh(["git", "apply", "--whitespace=nowarn", "-"], repo, inp=diff)
            c["applied"] = "git" if rc == 0 else None
            if rc != 0:
                rc2, _, _ = sh(["patch", "-p1", "--fuzz=3", "--no-backup-if-mismatch", "-s", "-f"], repo, inp=diff)
                if rc2 == 0:
                    c["applied"] = "patch_fuzz"
                else:
                    c["apply_error"] = err.strip().splitlines()[:2]
            if c["applied"]:
                rc, out, err = sh([sys.executable, str(REPO_EVAL), "t", ".", "--verify"], repo, env)
                try:
                    v = json.loads(out)
                    c.update({"clean": v["clean"], "changed": len(v["changed"]), "added": len(v["added"]),
                              "syntax_errors": v["syntax_errors"]})
                except Exception:
                    c["verify_error"] = (err or out)[-300:]
            row["candidates"].append(c)
            sh(["git", "reset", "--hard", "-q"], repo); sh(["git", "clean", "-fdq"], repo)
        return row
    except Exception as ex:
        row["error"] = repr(ex)
        return row
    finally:
        shutil.rmtree(tmp, ignore_errors=True)


def main():
    OUT.mkdir(parents=True, exist_ok=True)
    tasks = [json.loads(l) for l in open(DATA / "tasks.jsonl")]
    for t in tasks:
        t.pop("hints_text", None); t.pop("test_patch", None)
    if len(sys.argv) > 1:
        tasks = [t for t in tasks if t["instance_id"] in sys.argv[1:]]
    with Pool(int(os.environ.get("P4_PROCS", "4"))) as pool:
        rows = pool.map(run_task, tasks, chunksize=1)
    with open(OUT / "verify_per_task.jsonl", "w") as f:
        for r in rows:
            f.write(json.dumps(r, sort_keys=True) + "\n")
    s = {"tasks": len(rows), "errors": [r["instance_id"] for r in rows if "error" in r],
         "baseline_parity_ok": sum(1 for r in rows if r.get("baseline", {}).get("parity_ok")),
         "baseline_with_edit_sites": sum(1 for r in rows if r.get("baseline", {}).get("top_edit_sites"))}
    for name in ("gold", "e4b_O", "e4b_C"):
        cs = [c for r in rows for c in r["candidates"] if c["name"] == name]
        ap = [c for c in cs if c.get("applied")]
        s[name] = {"n": len(cs), "skipped": sum(1 for c in cs if c.get("skipped")),
                   "applied": len(ap), "applied_by_fuzz": sum(1 for c in ap if c["applied"] == "patch_fuzz"),
                   "clean": sum(1 for c in ap if c.get("clean")),
                   "syntax_rejected": sum(1 for c in ap if c.get("syntax_errors")),
                   "verify_error": sum(1 for c in ap if "verify_error" in c)}
    json.dump(s, open(OUT / "verify_summary.json", "w"), indent=1, sort_keys=True)
    print(json.dumps(s, indent=1, sort_keys=True))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/repro/phase4/adversarial_check.py
"""Adversarial contract checks for the production SemDAG retriever.

This is intentionally a standalone verification script rather than a harness
test file: scored task patches must never contain test-file modifications.
"""
from __future__ import annotations

import json
import os
from pathlib import Path
import sys
import tempfile

ROOT = Path(__file__).resolve().parents[2]
SEMDAG = Path(os.environ.get("P4_SEMDAG", ROOT / "AgenticSubmission/v4_semdag/skills/semdag/scripts"))
sys.path.insert(0, str(SEMDAG))

import cache  # noqa: E402
import repo_eval  # noqa: E402
import retriever  # noqa: E402
import semdag_ir_ast  # noqa: E402
import snapshot  # noqa: E402


def write(path: Path, text: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text)


def main():
    with tempfile.TemporaryDirectory(prefix="semdag_adversarial_") as td:
        root = Path(td) / "repo"
        write(root / "pkg/__init__.py", "from .util import helper\n\ndef entry(value):\n    return helper(value)\n")
        write(root / "pkg/util.py", "def helper(value):\n    return value + 1\n")
        write(root / "other.py", "def helper(value):\n    return value - 1\n")
        write(root / "test_pkg.py", "from pkg import entry\n\ndef test_entry():\n    assert entry(1) == 2\n")
        write(root / ".adk_exec_probe.py", "def wrapper():\n    return helper('value')\n")
        write(root / "semdag/internal.py", "def helper(value):\n    return 'tool-only'\n")
        write(root / "z_file.py", "def unrelated():\n    return 'noise'\n")

        os.environ["SEMDAG_CACHE"] = str(Path(td) / "cache")
        cache.init_container("fixture", str(root))
        snap = snapshot.snapshot("fixture", str(root))
        ir = semdag_ir_ast.build("fixture")
        parity = repo_eval.parity_check("fixture", snap, ir)
        assert parity["ok"], parity

        edges = {(e["src"], e["dst"], e["rel"], e.get("resolution")) for e in ir["edges"]}
        entry = next(n["id"] for n in ir["nodes"] if n.get("qualname") == "entry")
        pkg_helper = next(n["id"] for n in ir["nodes"] if n.get("qualname") == "helper" and n["file"] == "pkg/util.py")
        other_helper = next(n["id"] for n in ir["nodes"] if n.get("qualname") == "helper" and n["file"] == "other.py")
        assert (entry, pkg_helper, "calls", "from_import") in edges
        assert not any(s == entry and d == other_helper and rel == "calls" for s, d, rel, _ in edges)
        assert {"contains", "calls", "imports"} <= {e["rel"] for e in ir["edges"]}

        issue = "pkg.util helper called by entry should add one to value"
        first = retriever.rank(ir, snap, issue, top_n=5)
        second = retriever.rank(ir, snap, issue, top_n=5)
        assert first == second
        assert first["top"][0]["id"] == pkg_helper, first["top"]
        assert all(not retriever.is_test_path(site["file"]) for site in first["top"])
        assert all(not retriever.is_tool_path(site["file"]) for site in first["top"])
        assert {"contains", "calls", "imports"} <= set(first["relations_used"])

        lexical = retriever.rank(ir, snap, issue, top_n=5, damping=0.0)
        assert all(site["lexical_rank"] == i + 1 for i, site in enumerate(lexical["top"]))

        print(json.dumps({"ok": True, "parity": parity["ok"], "counts": ir["counts"],
                          "relations_used": first["relations_used"],
                          "top": [f"{s['file']}::{s['qualname']}" for s in first["top"]]}, indent=2))


if __name__ == "__main__":
    main()


## The lexical skill used in the paired agent runs (B1)

In [ ]:
%%writefile /kaggle/working/repro/locate.py
"""BM25 over function source, stdlib only. Usage: locate.py "<issue text>" [top_n]"""
import ast
import math
import os
import re
import sys
from collections import Counter

ROOT = "/workspace"
IDENT = re.compile(r"[A-Za-z_][A-Za-z0-9_]*")
CAMEL = re.compile(r"[A-Z]+(?=[A-Z][a-z])|[A-Z]?[a-z]+|[A-Z]+|\d+")
STOP = {"the", "and", "for", "that", "this", "with", "not", "but", "are", "was", "from", "have",
        "has", "when", "you", "can", "should", "would", "will", "use", "using", "def", "self",
        "return", "import", "none", "true", "false", "class", "if", "else", "in", "is", "of",
        "to", "it", "as", "on", "be", "or", "an", "at", "by", "we"}
SKIP = {".git", "node_modules", ".venv", "venv", "__pycache__", "site-packages", "build", "dist"}


def is_test(rel):
    parts = rel.split("/")
    name = parts[-1]
    return (any(p in ("tests", "test", "testing") for p in parts[:-1])
            or name.startswith("test_") or name.endswith("_test.py") or name == "conftest.py")


def toks(text):
    out = []
    for ident in IDENT.findall(text):
        low = ident.lower()
        if len(low) >= 2 and low not in STOP:
            out.append(low)
        parts = [p.lower() for chunk in ident.split("_") for p in CAMEL.findall(chunk)]
        if len(parts) > 1:
            out.extend(p for p in parts if len(p) >= 2 and p not in STOP)
    return out


def spans(src):
    out = []
    try:
        tree = ast.parse(src)
    except SyntaxError:
        return out

    def walk(node, prefix):
        for ch in ast.iter_child_nodes(node):
            if isinstance(ch, ast.ClassDef):
                walk(ch, prefix + [ch.name])
            elif isinstance(ch, (ast.FunctionDef, ast.AsyncFunctionDef)):
                start = min([ch.lineno] + [d.lineno for d in ch.decorator_list])
                out.append((start, ch.end_lineno, ".".join(prefix + [ch.name])))
                walk(ch, prefix + [ch.name])
            else:
                walk(ch, prefix)
    walk(tree, [])
    return out


def main():
    if len(sys.argv) < 2 or not sys.argv[1].strip():
        print("usage: locate.py \"<issue text>\" [top_n]")
        return
    query = Counter(toks(sys.argv[1]))
    top_n = int(sys.argv[2]) if len(sys.argv) > 2 and sys.argv[2].isdigit() else 8
    funcs = []
    for root, dirs, names in os.walk(ROOT):
        dirs[:] = sorted(d for d in dirs if d not in SKIP)
        for nm in sorted(names):
            if not nm.endswith(".py"):
                continue
            fp = os.path.join(root, nm)
            rel = os.path.relpath(fp, ROOT)
            if is_test(rel):
                continue
            try:
                src = open(fp, errors="replace").read()
            except OSError:
                continue
            lines = src.splitlines()
            for start, end, qual in spans(src):
                head = qual.replace(".", " ") + " " + rel.replace("/", " ").replace(".py", "")
                doc = Counter(toks("\n".join(lines[start - 1:end])) + toks(head) * 3)
                funcs.append((rel, start, end, qual, doc, sum(doc.values())))
    n = len(funcs)
    if not n:
        print("no functions found under /workspace")
        return
    df = Counter()
    for f in funcs:
        df.update(f[4].keys())
    avg = sum(f[5] for f in funcs) / n
    scores = [0.0] * n
    for term, qtf in query.items():
        d = df.get(term)
        if not d:
            continue
        w = math.log(1 + (n - d + 0.5) / (d + 0.5)) * min(qtf, 3)
        for i, f in enumerate(funcs):
            tf = f[4].get(term)
            if tf:
                scores[i] += w * tf * 2.2 / (tf + 1.2 * (0.25 + 0.75 * f[5] / avg))
    order = sorted(range(n), key=lambda i: (-scores[i], funcs[i][0], funcs[i][1]))
    print("Most likely edit sites (path:start-end qualified.name):")
    for i in order[:top_n]:
        if scores[i] <= 0:
            break
        rel, start, end, qual = funcs[i][:4]
        print(f"{rel}:{start}-{end} {qual}")


main()


## Run

In [ ]:
!cd /kaggle/working/repro/phase4 && python measure.py > ../out/run.log 2>&1; tail -1 ../out/run.log
!cd /kaggle/working/repro/phase4 && python measure_v2.py > ../out/v2_run.log 2>&1; tail -1 ../out/v2_run.log
!cd /kaggle/working/repro/phase4 && P4_KEEP_TESTS=1 P4_OUT=/kaggle/working/repro/out_keep_tests python measure_v2.py > ../out/v2_keep_run.log 2>&1; tail -1 ../out/v2_keep_run.log
!cd /kaggle/working/repro/phase4 && python equal_footing.py > ../out/equal_footing.log 2>&1; tail -1 ../out/equal_footing.log
!cd /kaggle/working/repro/phase4 && python verify_gate.py > ../out/verify_run.log 2>&1; tail -1 ../out/verify_run.log
!cd /kaggle/working/repro/phase4 && P4_SEMDAG=/kaggle/working/repro/legacy_semdag P4_OUT=/kaggle/working/repro/out_legacy_gate python verify_gate.py > ../out/verify_legacy_run.log 2>&1; tail -1 ../out/verify_legacy_run.log
!cd /kaggle/working/repro/phase4 && python adversarial_check.py > ../out/adversarial_check.log 2>&1; echo adversarial exit $?

## Check against the paper

In [ ]:
import json
W = '/kaggle/working/repro'
O = f'{W}/out'
s1 = json.load(open(f'{O}/summary.json')); s2 = json.load(open(f'{O}/v2_summary.json'))
sv = json.load(open(f'{O}/verify_summary.json')); sl = json.load(open(f'{W}/out_legacy_gate/verify_summary.json'))
ef = json.load(open(f'{O}/equal_footing.json')); sk = json.load(open(f'{W}/out_keep_tests/v2_summary.json'))
rows = [json.loads(l) for l in open(f'{O}/per_task.jsonl')]
diag = [r['ppr_diag']['A'] for r in rows if r.get('gold') and 'A' in r.get('ppr_diag', {})]
f = lambda t, a: [t[a][f'fn@{k}'] for k in (1, 5, 10)]
checks = {
 'Table 1: gold functions': (s1['gold_fns'], 438),
 'Table 1: present in shipped graph': (s1['shipped']['gold_present'], 376),
 'Table 1: async present in shipped graph': (s1['shipped']['gold_present_async'], 0),
 'Table 2: A': ([s1['arms']['A'][f'fn_hit@{k}'] for k in (1, 5, 10)] + [s1['arms']['A']['file_hit@10']], [13, 25, 34, 64]),
 'Table 2: Af': ([s1['arms']['Af'][f'fn_hit@{k}'] for k in (1, 5, 10)] + [s1['arms']['Af']['file_hit@10']], [13, 28, 36, 75]),
 'Table 2: B': ([s1['arms']['B'][f'fn_hit@{k}'] for k in (1, 5, 10)] + [s1['arms']['B']['file_hit@10']], [7, 18, 22, 35]),
 'A rank mass on tests': (ef['ppr_A']['mean_mass_on_test_functions'], 0.402),
 'A top-five slots that are tests': (sum(d['test_fns_in_top5'] for d in diag), 106),
 'A tasks tied at rank 5': (sum(d['tied_at_rank5'] for d in diag), 53),
 'legacy gate: parity ok on full snapshots': (sl['baseline_parity_ok'], 0),
 'legacy gate: parity ok, .py-only layout': (s1['parity_ok'], 1),
 'production gate: parity ok': (sv['baseline_parity_ok'], 129),
 'production gate: candidates returned': (sv['baseline_with_edit_sites'], 129),
 'gold patches judged clean': (sv['gold']['clean'], 129),
 'selection winner': (s2['winner'], 'X'),
 'Table 3: X selection/held-out/all': (f(s2['selection'], 'X') + f(s2['heldout_rich'], 'X') + f(s2['all'], 'X') + [s2['all']['X']['file@10']], [18, 38, 44, 3, 20, 24, 21, 58, 68, 108]),
 'Table 3: X tests kept': (f(sk['selection'], 'X') + f(sk['heldout_rich'], 'X') + f(sk['all'], 'X') + [sk['all']['X']['file@10']], [14, 26, 33, 4, 11, 14, 18, 37, 47, 82]),
 'Table 3: NX': (f(s2['selection'], 'NX') + f(s2['heldout_rich'], 'NX') + f(s2['all'], 'NX') + [s2['all']['NX']['file@10']], [15, 30, 34, 10, 16, 24, 25, 46, 58, 103]),
 'Table 3: D15': (f(s2['selection'], 'D15') + f(s2['heldout_rich'], 'D15') + f(s2['all'], 'D15') + [s2['all']['D15']['file@10']], [15, 36, 43, 4, 20, 26, 19, 56, 69, 111]),
 'Table 3: D50': (f(s2['selection'], 'D50') + f(s2['heldout_rich'], 'D50') + f(s2['all'], 'D50') + [s2['all']['D50']['file@10']], [10, 22, 40, 6, 23, 28, 16, 45, 68, 110]),
 'Table 3: DC85': (f(s2['selection'], 'DC85') + f(s2['heldout_rich'], 'DC85') + f(s2['all'], 'DC85') + [s2['all']['DC85']['file@10']], [8, 24, 41, 3, 13, 21, 11, 37, 62, 109]),
 'D30 Hit@5 all': (s2['all']['D30']['fn@5'], 52),
 'X - Af Hit@5, all': (ef['X-Af fn@5 all (both filtered)']['diff'], 0.2586),
 'X - A Hit@5, tests kept, all': (ef['Xall-A fn@5 all (both unfiltered)']['diff'], 0.1034),
 'D15 - X Hit@5, all': (s2['dag_contrasts']['D15-X fn@5 all']['diff'], -0.0172),
 'Table 4: BM25 on shipped text': ([s2['shipped_bm25']['all'][f'fn@{k}'] for k in (1, 5, 10)], [21, 54, 61]),
 'Table 4: async-gold tasks': (s2['async_gold_tasks']['X'][1:] + s2['async_gold_tasks']['shipped_bm25'][1:], [12, 15, 7, 9]),
}
bad = 0
for k, (got, want) in checks.items():
    ok = got == want
    bad += not ok
    print(('OK  ' if ok else 'DIFF'), k, got, '' if ok else f'expected {want}')
print(f'{len(checks) - bad} of {len(checks)} checks match the paper')
print(json.dumps(ef, indent=1)); print(json.dumps(s2['dag_contrasts'], indent=1))